In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║ VG360 · Clifford-Geometry Industrial Safety Analytics — CPU-only, single cell ║
# ║                                                                              ║
# ║  gray frame → YOLOX-Nano (ONNX, int8) → person boxes                         ║
# ║    → PGA  Cl(3,0,1): pixel ray ∨ camera, ∧ ground plane → world xyz          ║
# ║    → CGA  Cl(4,1)  : P·S zone test (spheres / half-spaces), 2D-PGA polygons  ║
# ║    → motors Cl(2,0,1): constant-velocity tracker on SE(2)                    ║
# ║    → rotor  Cl(3,0) : tilt of fitted 3D body pose → fall rule                ║
# ║    → PPE head classifier: Clifford conv Cl(2,0) + SAM, folded to plain conv  ║
# ║    → per-event MCC  ·  Lean4 proofs  ·  OpenAI-compatible /v1/chat/completions║
# ║                                                                              ║
# ║ Principle: derive in GA, deploy closed form. Every GA derivation is asserted ║
# ║ equal to the numpy closed form used at runtime, and the closed forms are the ║
# ║ statements proved in Lean.                                                   ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ─────────────────────────────── §0 setup & config ───────────────────────────────
import sys, subprocess, importlib, os, json, time, math, threading, uuid, re, shutil, random
from pathlib import Path
from dataclasses import dataclass, field

import importlib.util
def _pip(**pkgs):                     # import_name → pip_name
    missing = [pip for mod, pip in pkgs.items() if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

_pip(onnxruntime="onnxruntime", onnx="onnx", fastapi="fastapi", uvicorn="uvicorn", sklearn="scikit-learn",
     scipy="scipy", cv2="opencv-python-headless", requests="requests")

import numpy as np, cv2, requests
import torch, torch.nn as nn, torch.nn.functional as F
import onnxruntime as ort
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import matthews_corrcoef, confusion_matrix

IN_COLAB = "google.colab" in sys.modules
BASE = Path("/content/vg360") if Path("/content").exists() else Path.cwd() / "vg360"
BASE.mkdir(parents=True, exist_ok=True)
SEED = 7
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(max(1, os.cpu_count() or 1))

@dataclass
class Config:
    # detector
    detector_url: str = "https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_nano.onnx"
    detector_onnx: str = ""            # set to your own YOLOX-Nano gray ONNX to override (COCO class 0 = person assumed)
    det_size: int = 416
    det_conf: float = 0.35
    det_nms: float = 0.45
    quantize_int8: bool = True         # static QDQ int8 via onnxruntime, calibrated on your frames
    ort_threads: int = max(1, (os.cpu_count() or 2))
    # input
    source: str = ""                   # video file / RTSP / image of YOUR site; empty → synthetic oil-rig demo clip
    upload_photo: bool = False         # Colab: True → file picker for a rig photo/clip, used as `source`
    lens: dict = None                  # {'K','dist'} or {'model':'fisheye','K','D','balance'} at native resolution → frames are undistorted
    fisheye_demo: bool = True          # §9b: convert the rig demo to a 185° fisheye and run it through virtual views
    scene: str = "rig_elevated"        # preset for your own footage: "rig_elevated" | "street_demo" (see SCENES)
    tiles: int = 3                     # elevated cameras see small people: full frame + tiles×tiles overlapping crops (0 = off)
    auto_height: bool = True           # re-estimate camera height from standing workers in the first frame
    demo_image_url: str = "https://raw.githubusercontent.com/ultralytics/yolov5/master/data/images/bus.jpg"
    max_frames: int = 300
    # camera (overwritten by auto-calibration if you call calibrate_from_vanishing_points)
    img_w: int = 1280; img_h: int = 720
    focal_px: float = 900.0
    cam_height_m: float = 6.0
    pitch_deg: float = 35.0            # downward tilt
    yaw_deg: float = 0.0
    # geometry backend at runtime: "fast" (closed form) or "pga" (engine, slower, identical result)
    geom_backend: str = "fast"
    # tracker / rules
    gate_m: float = 2.5
    max_age_s: float = 1.5
    blend: float = 0.6                 # geodesic blend toward measurement on SE(2)
    vel_alpha: float = 0.4
    fall_head_z: float = 0.8
    fall_tilt_deg: float = 60.0
    fall_sustain_s: float = 1.0
    stand_head_z: float = 1.2
    intrusion_sustain_s: float = 0.3
    rule_gap_s: float = 0.5            # Rule Studio engine: tolerated detection dropout inside a rule's duration
    # PPE classifier
    ppe_dir: str = ""                  # ImageFolder with subfolders helmet/ no_helmet/ (see voc_to_crops for SHWD)
    ppe_size: int = 48
    ppe_epochs: int = 14
    sam_rho: float = 0.05
    # simulation for MCC of geometric stages
    sim_people: int = 14
    sim_seconds: float = 60.0
    sim_fps: float = 10.0
    # Lean
    run_lean: bool = False             # True → installs elan + Mathlib cache in Colab (~several minutes) and checks proofs
    # OpenAI-compatible endpoint
    serve: bool = True
    port: int = 8000
    upstream_base_url: str = os.environ.get("UPSTREAM_BASE_URL", "")   # optional real LLM behind the gateway
    upstream_api_key: str = os.environ.get("UPSTREAM_API_KEY", "")
    upstream_model: str = os.environ.get("UPSTREAM_MODEL", "")

CFG = Config()
REPORT = {}          # everything measured lands here
EVENT_LOG = []       # safety events (dicts), consumed by the endpoint
print(f"[§0] base={BASE}  torch={torch.__version__}  ort={ort.__version__}  threads={CFG.ort_threads}")

# ─────────────────────────────── §1 Lean4 proofs ─────────────────────────────────
# The runtime closed forms below are exactly these statements.
LEAN_SRC = r'''
import Mathlib

/-! Conformal model over ℝ³ ⊕ ⟨e∞, e₀⟩ with e₀·e∞ = -1, e₀² = e∞² = 0.
    A conformal vector is (x₁,x₂,x₃, a, b) meaning x + a·e∞ + b·e₀. -/
def cdot (x1 x2 x3 a b y1 y2 y3 c d : ℝ) : ℝ :=
  x1*y1 + x2*y2 + x3*y3 - (a*d + b*c)

/-- up(x)·up(y) = -½‖x-y‖²  (point–point distance) -/
theorem cga_dist (x1 x2 x3 y1 y2 y3 : ℝ) :
    cdot x1 x2 x3 ((x1^2+x2^2+x3^2)/2) 1 y1 y2 y3 ((y1^2+y2^2+y3^2)/2) 1
      = -((x1-y1)^2 + (x2-y2)^2 + (x3-y3)^2)/2 := by
  unfold cdot; ring

/-- up(x)·(up(c) - ½r²e∞) = ½(r² - ‖x-c‖²): positive ⇔ inside the sphere -/
theorem cga_sphere (x1 x2 x3 c1 c2 c3 r : ℝ) :
    cdot x1 x2 x3 ((x1^2+x2^2+x3^2)/2) 1 c1 c2 c3 ((c1^2+c2^2+c3^2)/2 - r^2/2) 1
      = (r^2 - ((x1-c1)^2 + (x2-c2)^2 + (x3-c3)^2))/2 := by
  unfold cdot; ring

theorem cga_sphere_inside (x1 x2 x3 c1 c2 c3 r : ℝ) :
    0 < cdot x1 x2 x3 ((x1^2+x2^2+x3^2)/2) 1 c1 c2 c3 ((c1^2+c2^2+c3^2)/2 - r^2/2) 1
      ↔ (x1-c1)^2 + (x2-c2)^2 + (x3-c3)^2 < r^2 := by
  rw [cga_sphere]; constructor <;> intro h <;> linarith

/-- up(x)·(n + δ e∞) = n·x - δ  (signed distance to a plane, ‖n‖ = 1) -/
theorem cga_plane (x1 x2 x3 n1 n2 n3 δ : ℝ) :
    cdot x1 x2 x3 ((x1^2+x2^2+x3^2)/2) 1 n1 n2 n3 δ 0 = n1*x1 + n2*x2 + n3*x3 - δ := by
  unfold cdot; ring

/-- PGA meet of ray C + t·v with plane n·X + δ = 0: any t with t(n·v) = -(n·C+δ) lies on the plane -/
theorem ray_plane (n1 n2 n3 δ c1 c2 c3 v1 v2 v3 t : ℝ)
    (ht : t * (n1*v1 + n2*v2 + n3*v3) = -(n1*c1 + n2*c2 + n3*c3 + δ)) :
    n1*(c1 + t*v1) + n2*(c2 + t*v2) + n3*(c3 + t*v3) + δ = 0 := by
  linear_combination ht

/-- the closed-form t used at runtime satisfies the hypothesis of `ray_plane` when n·v ≠ 0 -/
theorem ray_plane_t (nv k : ℝ) (h : nv ≠ 0) : (-k / nv) * nv = -k := by
  field_simp

/-! Rule Studio properties, stated over the engine's own model (RuleEngine in §7b). -/
inductive Pri | P1 | P2 | P3 deriving DecidableEq

/-- the engine emits an alert iff it fired and (it is P1 or not within the cooldown) -/
def emitted (p : Pri) (alert recent : Bool) : Bool := alert && (p == Pri.P1 || !recent)

/-- debouncing never suppresses a P1 alert -/
theorem debounce_keeps_p1 (alert recent : Bool) (h : alert = true) : emitted Pri.P1 alert recent = true := by
  subst h; cases recent <;> rfl

/-- every emitted alert is written to the ledger: appending puts the entry in it -/
def record {α : Type} (ledger : List α) (e : α) : List α := ledger ++ [e]
theorem alert_has_evidence {α : Type} (ledger : List α) (e : α) : e ∈ record ledger e := by
  simp [record]

/-- 2D-PGA side test: the line through a,b evaluated at p is the 2D cross product (polygon zones) -/
theorem side_of_line (a1 a2 b1 b2 p1 p2 : ℝ) :
    (a2 - b2)*p1 + (b1 - a1)*p2 + (a1*b2 - a2*b1)
      = (b1 - a1)*(p2 - a2) - (b2 - a2)*(p1 - a1) := by
  ring
'''
LEAN_DIR = BASE / "lean"; LEAN_DIR.mkdir(exist_ok=True)
(LEAN_DIR / "CGA.lean").write_text(LEAN_SRC)

def check_lean():
    """Colab only: elan + Mathlib (cached oleans) → `lake env lean CGA.lean`."""
    home = Path.home(); elan = home / ".elan/bin"
    env = dict(os.environ, PATH=f"{elan}:{os.environ['PATH']}")
    if not (elan / "lake").exists():
        subprocess.run("curl -sSfL https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh -o /tmp/elan-init.sh"
                       " && sh /tmp/elan-init.sh -y --default-toolchain none", shell=True, check=True)
    proj = LEAN_DIR / "cga_proofs"
    if not proj.exists():
        subprocess.run("lake +leanprover-community/mathlib4:lean-toolchain new cga_proofs math",
                       shell=True, check=True, cwd=LEAN_DIR, env=env)
        subprocess.run("lake exe cache get", shell=True, check=True, cwd=proj, env=env)
    shutil.copy(LEAN_DIR / "CGA.lean", proj / "CGA.lean")
    r = subprocess.run("lake env lean CGA.lean", shell=True, cwd=proj, env=env, capture_output=True, text=True)
    ok = r.returncode == 0
    print("[§1] Lean:", "all theorems checked ✓" if ok else "FAILED\n" + r.stdout + r.stderr)
    return ok

REPORT["lean"] = check_lean() if CFG.run_lean else "not run (set CFG.run_lean=True in Colab)"
print(f"[§1] Lean source → {LEAN_DIR/'CGA.lean'}  status: {REPORT['lean']}")

# ─────────────────────────────── §2 Clifford engine ──────────────────────────────
class Cl:
    """Generic Cl(p,q,r) on bitmask blades. metric[i] = e_i² ∈ {+1,-1,0}."""
    def __init__(self, metric, names):
        self.metric, self.names, self.n = list(metric), list(names), len(metric)
        self.N = 1 << self.n
        N = self.N
        self.gp_idx = np.zeros((N, N), int); self.gp_sgn = np.zeros((N, N))
        self.op_sgn = np.zeros((N, N))
        for a in range(N):
            for b in range(N):
                s = self._reorder(a, b)
                m = s
                for i in range(self.n):
                    if (a & b) >> i & 1: m *= self.metric[i]
                self.gp_idx[a, b] = a ^ b; self.gp_sgn[a, b] = m
                self.op_sgn[a, b] = s if (a & b) == 0 else 0
        self.grade = np.array([bin(m).count("1") for m in range(N)])
        full = N - 1
        D = np.zeros((N, N))                 # right-complement dual (metric-free → works for degenerate PGA)
        for m in range(N):
            D[full ^ m, m] = self._reorder(m, full ^ m)
        self.D, self.Dinv = D, np.linalg.inv(D)
        self.rev_sgn = np.array([(-1) ** (g * (g - 1) // 2) for g in self.grade])

    @staticmethod
    def _reorder(a, b):
        a >>= 1; s = 0
        while a: s += bin(a & b).count("1"); a >>= 1
        return -1.0 if s & 1 else 1.0

    def mv(self, **coefs):
        v = np.zeros(self.N)
        for k, c in coefs.items():
            if k in ("s", "scalar"): v[0] += c; continue
            k = k[1:]                         # 'e012' → '012'
            m = 0; sgn = 1.0
            for ch in k:
                i = self.names.index(ch); sgn *= self._reorder(m, 1 << i); m |= 1 << i
            v[m] += sgn * c
        return v

    def _prod(self, a, b, sgn):
        out = np.zeros(self.N)
        np.add.at(out, self.gp_idx.ravel(), (sgn * np.outer(a, b)).ravel())
        return out
    def gp(self, a, b): return self._prod(a, b, self.gp_sgn)
    def op(self, a, b): return self._prod(a, b, self.op_sgn)
    def rp(self, a, b): return self.Dinv @ self.op(self.D @ a, self.D @ b)    # regressive (join)
    def rev(self, a): return a * self.rev_sgn
    def sand(self, M, X): return self.gp(self.gp(M, X), self.rev(M))
    def scalar(self, a, b): return self.gp(a, b)[0]
    def exp(self, B, terms=30):
        out = np.zeros(self.N); out[0] = 1; term = out.copy()
        for k in range(1, terms):
            term = self.gp(term, B) / k; out = out + term
        return out

PGA3 = Cl([0, 1, 1, 1], ["0", "1", "2", "3"])     # projective 3D
PGA2 = Cl([0, 1, 1], ["0", "1", "2"])             # projective 2D (image plane, ground plane)
CGA = Cl([1, 1, 1, 1, -1], ["1", "2", "3", "p", "m"])   # conformal 3D (e+, e-)
CL2 = Cl([1, 1], ["1", "2"])                      # Cl(2,0) for Clifford convolutions
CL3 = Cl([1, 1, 1], ["1", "2", "3"])              # Cl(3,0) rotors

# ─────────────────────────────── §3 PGA camera geometry ──────────────────────────
# PGA3: plane a·x+b·y+c·z+d=0 ↦ a e1 + b e2 + c e3 + d e0 ; point = ∧ of three planes
def pga_plane(a, b, c, d): return PGA3.mv(e1=a, e2=b, e3=c, e0=d)
def pga_point(x, y, z):
    return PGA3.op(PGA3.op(pga_plane(1, 0, 0, -x), pga_plane(0, 1, 0, -y)), pga_plane(0, 0, 1, -z))
def pga_ideal(dx, dy, dz):                     # point at infinity (direction): limit of point(t·d)/t
    return pga_point(dx, dy, dz) - pga_point(0, 0, 0)
_E123, _E023, _E013, _E012 = (PGA3.mv(e123=1), PGA3.mv(e023=1), PGA3.mv(e013=1), PGA3.mv(e012=1))
def pga_coords(P):
    w = P @ _E123
    return np.array([P @ _E023, P @ _E013, P @ _E012]) / w * np.array([-1, 1, -1])
# 2D: line a·x+b·y+c=0 ↦ a e1 + b e2 + c e0 ; point = ∧ of two lines
def pga2_line(a, b, c): return PGA2.mv(e1=a, e2=b, e0=c)
def pga2_point(x, y): return PGA2.op(pga2_line(1, 0, -x), pga2_line(0, 1, -y))
_F12, _F02, _F01 = PGA2.mv(e12=1), PGA2.mv(e02=1), PGA2.mv(e01=1)
def pga2_coords(P):
    w = P @ _F12
    if abs(w) < 1e-12: return None             # ideal point (vanishing direction at infinity)
    return np.array([-(P @ _F02), (P @ _F01)]) / w

@dataclass
class Camera:
    f: float; cx: float; cy: float; h: float; pitch: float; yaw: float
    roll: float = 0.0; x0: float = 0.0; y0: float = 0.0     # image-plane roll and the camera's deck position
    def __post_init__(self):
        th, ps, rl = math.radians(self.pitch), math.radians(self.yaw), math.radians(self.roll)
        fwd = np.array([math.sin(ps) * math.cos(th), math.cos(ps) * math.cos(th), -math.sin(th)])
        right = np.array([fwd[1], -fwd[0], 0.0]); right /= np.linalg.norm(right)
        down = np.cross(fwd, right)
        right, down = math.cos(rl) * right + math.sin(rl) * down, -math.sin(rl) * right + math.cos(rl) * down
        self.R = np.stack([right, down, fwd], 1)          # cam → world
        self.C = np.array([self.x0, self.y0, self.h])
        self.Kinv = np.linalg.inv(np.array([[self.f, 0, self.cx], [0, self.f, self.cy], [0, 0, 1.0]]))
    def ray(self, u, v):
        return self.R @ (self.Kinv @ np.array([u, v, 1.0]))
    def project(self, X):
        Xc = (np.atleast_2d(X) - self.C) @ self.R
        return np.stack([self.f * Xc[:, 0] / Xc[:, 2] + self.cx, self.f * Xc[:, 1] / Xc[:, 2] + self.cy], 1)

# --- PGA derivation (engine) ---
def ground_point_pga(cam, u, v, plane=(0, 0, 1, 0)):
    L = PGA3.rp(pga_point(*cam.C), pga_ideal(*cam.ray(u, v)))     # camera ∨ point-at-infinity = ray
    return pga_coords(PGA3.op(pga_plane(*plane), L))              # plane ∧ line = point
# --- closed form (deployed; proved as `ray_plane`) ---
def ray_plane_fast(C, d, n, delta):
    nd = n @ d
    if abs(nd) < 1e-12: return None
    return C + (-(n @ C + delta) / nd) * d
def ground_point(cam, u, v):
    if CFG.geom_backend == "pga": return ground_point_pga(cam, u, v)
    return ray_plane_fast(cam.C, cam.ray(u, v), np.array([0, 0, 1.0]), 0.0)
def head_height(cam, u_top, v_top, foot):
    """head ray ∧ vertical plane through the foot point facing the camera → z of head."""
    n = foot - cam.C; n[2] = 0; n /= (np.linalg.norm(n) + 1e-9)
    if CFG.geom_backend == "pga":
        return ground_point_pga(cam, u_top, v_top, plane=(n[0], n[1], 0, -(n @ foot)))[2]
    X = ray_plane_fast(cam.C, cam.ray(u_top, v_top), n, -(n @ foot))
    return X[2] if X is not None else 0.0

# --- auto-calibration from straight edges (Lehar: parallel lines meet at infinity) ---
def vanishing_point(segments):
    """segments: list of ((x1,y1),(x2,y2)) image segments of parallel world lines → least-squares VP via 2D PGA."""
    lines = [PGA2.rp(pga2_point(*a), pga2_point(*b)) for a, b in segments]
    A = np.array([[l @ PGA2.mv(e1=1), l @ PGA2.mv(e2=1), l @ PGA2.mv(e0=1)] for l in lines])
    A /= np.linalg.norm(A[:, :2], axis=1, keepdims=True)
    _, _, Vt = np.linalg.svd(A); x = Vt[-1]
    return None if abs(x[2]) < 1e-12 else x[:2] / x[2]
def calibrate_from_vanishing_points(v1, v2, cx, cy):
    """two VPs of orthogonal horizontal directions (zero roll) → focal length and pitch."""
    p = np.array([cx, cy]); f2 = -np.dot(np.asarray(v1) - p, np.asarray(v2) - p)
    f = math.sqrt(max(f2, 1e-9))
    hz = PGA2.rp(pga2_point(*v1), pga2_point(*v2))                       # horizon line = join of VPs
    a, b, c = hz @ PGA2.mv(e1=1), hz @ PGA2.mv(e2=1), hz @ PGA2.mv(e0=1)
    v_h = -(a * cx + c) / b                                               # horizon height at x = cx
    return f, math.degrees(math.atan2(cy - v_h, f))
def calibrate_height_from_people(cam_kw, boxes, assumed_height=1.70):
    """height scales every metric estimate linearly → fix it from median standing-person height."""
    probe = Camera(**{**cam_kw, "h": 1.0}); hs = []
    for x1, y1, x2, y2 in boxes:
        ft = ground_point(probe, (x1 + x2) / 2, y2)
        if ft is not None: hs.append(head_height(probe, (x1 + x2) / 2, y1, ft))
    return assumed_height / float(np.median(hs))


# --- direct calibration from known deck points (homography → K, pose, roll), with optional lens undistortion ---
def camera_from_krt(K, R_wc, C):
    """Camera with intrinsics K, world→camera rotation R_wc and centre C (world, z up)."""
    cam = Camera(float(K[0, 0]), float(K[0, 2]), float(K[1, 2]), float(C[2]), 0.0, 0.0)
    cam.R = R_wc.T.copy(); cam.C = np.asarray(C, float).copy(); cam.Kinv = np.linalg.inv(K)
    fwd = cam.R[:, 2]; rl = np.array([fwd[1], -fwd[0], 0.0]); rl /= np.linalg.norm(rl); dl = np.cross(fwd, rl)
    cam.pitch = math.degrees(math.asin(-fwd[2])); cam.yaw = math.degrees(math.atan2(fwd[0], fwd[1]))
    cam.roll = math.degrees(math.atan2(cam.R[:, 0] @ dl, cam.R[:, 0] @ rl)); cam.x0, cam.y0 = C[0], C[1]
    return cam

def calibrate_from_points(image_pts, world_pts, img_size, K=None, dist=None, max_px=3.0, max_m=0.15):
    """≥4 deck points with known (x, y) metres and their pixels → camera. Solves the ground homography H,
    recovers focal length (if K unknown: square pixels, centred principal point), height, pitch, yaw, roll and
    position, then reports reprojection error in pixels and back-projection error on the deck in metres."""
    ip = np.asarray(image_pts, np.float64).reshape(-1, 2); wp = np.asarray(world_pts, np.float64).reshape(-1, 2)
    if len(ip) < 4 or len(ip) != len(wp): raise ValueError("need ≥ 4 matching image/world points")
    W, H = img_size
    if dist is not None:
        if K is None: raise ValueError("lens distortion needs the lens matrix K")
        K = np.asarray(K, float); ip = cv2.undistortPoints(ip.reshape(-1, 1, 2), K, np.asarray(dist, float), P=K).reshape(-1, 2)
    Hm, mask = cv2.findHomography(wp, ip, cv2.RANSAC if len(ip) >= 6 else 0, 3.0)
    if Hm is None: raise ValueError("points are degenerate (collinear or repeated)")
    inl = mask.ravel().astype(bool) if mask is not None else np.ones(len(ip), bool)
    estimate_f = K is None
    if estimate_f:
        T = np.array([[1, 0, -W / 2], [0, 1, -H / 2], [0, 0, 1.0]]); Hc = T @ Hm; h1, h2 = Hc[:, 0], Hc[:, 1]; est = []
        if abs(h1[2] * h2[2]) > 1e-12: est.append(-(h1[0] * h2[0] + h1[1] * h2[1]) / (h1[2] * h2[2]))     # r1 ⟂ r2
        if abs(h2[2] ** 2 - h1[2] ** 2) > 1e-12: est.append((h1[0] ** 2 + h1[1] ** 2 - h2[0] ** 2 - h2[1] ** 2) / (h2[2] ** 2 - h1[2] ** 2))  # ‖r1‖=‖r2‖
        est = [e for e in est if e > 0]
        f = math.sqrt(float(np.median(est))) if est else float(max(W, H))
        K = np.array([[f, 0, W / 2], [0, f, H / 2], [0, 0, 1.0]])
    obj = np.c_[wp, np.zeros(len(wp))].astype(np.float64)
    def pose(Kf):                                         # planar PnP (IPPE) on the inliers for a given K
        ok, rv, tv = cv2.solvePnP(obj[inl], ip[inl], Kf, None, flags=cv2.SOLVEPNP_IPPE)
        if not ok: return None, None, 1e9
        R_ = cv2.Rodrigues(rv)[0]; t_ = tv.ravel(); pr = (Kf @ (R_ @ obj[inl].T + t_[:, None])).T
        return R_, t_, float(np.sqrt(np.mean(np.sum((pr[:, :2] / pr[:, 2:] - ip[inl]) ** 2, 1))))
    if estimate_f:                                        # refine f: homography guess, then 1-D search on reprojection error
        mk = lambda f_: np.array([[f_, 0, W / 2], [0, f_, H / 2], [0, 0, 1.0]])
        grid = np.geomspace(0.4 * f, 2.5 * f, 60); errs = [pose(mk(g))[2] for g in grid]; i0 = int(np.argmin(errs))
        lo, hi = grid[max(i0 - 1, 0)], grid[min(i0 + 1, len(grid) - 1)]
        for _ in range(40):                               # golden-section refinement
            m1, m2 = lo + 0.382 * (hi - lo), lo + 0.618 * (hi - lo)
            if pose(mk(m1))[2] < pose(mk(m2))[2]: hi = m2
            else: lo = m1
        K = mk((lo + hi) / 2)
    K = np.asarray(K, float); R, t, _ = pose(K)
    if R is None: raise ValueError("pose could not be solved from these points")
    C = -R.T @ t
    warn = []
    if C[2] <= 0: warn.append("camera comes out below the deck: check that world x, y form a right-handed frame (x east, y north)")
    cam = camera_from_krt(K, R, C)
    rep = cam.project(np.c_[wp, np.zeros(len(wp))]); px = np.linalg.norm(rep - ip, axis=1)
    gp = np.array([ray_plane_fast(cam.C, cam.ray(u, v), np.array([0, 0, 1.0]), 0.0)[:2] for u, v in ip]); m = np.linalg.norm(gp - wp, axis=1)
    rms_px, rms_m = float(np.sqrt(np.mean(px[inl] ** 2))), float(np.sqrt(np.mean(m[inl] ** 2)))
    if rms_px > max_px: warn.append(f"reprojection error {rms_px:.1f} px > {max_px} px: re-click points or add lens distortion")
    if rms_m > max_m: warn.append(f"deck error {rms_m:.2f} m > {max_m} m: spread points wider and nearer the zones you care about")
    if (~inl).any(): warn.append(f"{int((~inl).sum())} point(s) rejected as outliers")
    return {"camera": cam, "H": Hm, "f_px": float(K[0, 0]), "height_m": float(C[2]), "pitch_deg": cam.pitch, "yaw_deg": cam.yaw,
            "roll_deg": cam.roll, "position_m": (float(C[0]), float(C[1])), "rms_px": rms_px, "rms_m": rms_m,
            "per_point_m": [round(float(v), 3) for v in m], "inliers": int(inl.sum()), "n": len(ip), "warnings": warn}


# --- fisheye lenses (Kannala–Brandt, as OpenCV's cv2.fisheye) and virtual pinhole views ---
from types import SimpleNamespace
def make_fisheye(K, D, R_cw, C, size):
    """fisheye camera: K (3×3), D = [k1..k4], R_cw camera→world (columns right, down, forward), centre C, size (W, H)."""
    return SimpleNamespace(K=np.asarray(K, float), D=np.asarray(D, float).ravel(), R_cw=np.asarray(R_cw, float), C=np.asarray(C, float), size=size)
def kb_dir_to_px(fish, dc):
    """camera-frame directions (N,3) → fisheye pixels; θ = angle from the optical axis, θd = θ(1+k1θ²+k2θ⁴+k3θ⁶+k4θ⁸)."""
    x, y, z = dc[..., 0], dc[..., 1], dc[..., 2]; r = np.hypot(x, y); th = np.arctan2(r, z); k = fish.D
    thd = th * (1 + k[0] * th ** 2 + k[1] * th ** 4 + k[2] * th ** 6 + k[3] * th ** 8)
    s = np.where(r > 1e-12, thd / np.maximum(r, 1e-12), 0.0)
    return np.stack([fish.K[0, 0] * x * s + fish.K[0, 2], fish.K[1, 1] * y * s + fish.K[1, 2]], -1), th
def kb_project(fish, Xw):
    return kb_dir_to_px(fish, (np.atleast_2d(Xw) - fish.C) @ fish.R_cw)
def kb_ray(fish, uv):
    """fisheye pixels → world ray directions (Newton inversion of θd)."""
    uv = np.atleast_2d(uv).astype(float); mx = (uv[:, 0] - fish.K[0, 2]) / fish.K[0, 0]; my = (uv[:, 1] - fish.K[1, 2]) / fish.K[1, 1]
    thd = np.hypot(mx, my); th = thd.copy(); k = fish.D
    for _ in range(20):
        f_ = th * (1 + k[0] * th ** 2 + k[1] * th ** 4 + k[2] * th ** 6 + k[3] * th ** 8) - thd
        df = 1 + 3 * k[0] * th ** 2 + 5 * k[1] * th ** 4 + 7 * k[2] * th ** 6 + 9 * k[3] * th ** 8; th = th - f_ / df
    phi = np.arctan2(my, mx); dc = np.stack([np.sin(th) * np.cos(phi), np.sin(th) * np.sin(phi), np.cos(th)], -1)
    return dc @ fish.R_cw.T
def fisheye_ground(fish, uv, z=0.0):
    d = kb_ray(fish, uv); t = (z - fish.C[2]) / np.where(np.abs(d[:, 2]) < 1e-9, -1e-9, d[:, 2])
    out = fish.C[None, :2] + t[:, None] * d[:, :2]; out[t <= 0] = np.nan; return out
def fisheye_views(fish, specs, size=(1280, 720), hfov=60.0):
    """one fisheye → upright virtual pinhole cameras. specs: [(yaw_deg, pitch_deg), …] in world terms.
    Each view is an ordinary Camera (zero roll, so people stand upright) plus a remap from the fisheye frame."""
    w, h = size; f = (w / 2) / math.tan(math.radians(hfov / 2)); out = []
    uu, vv = np.meshgrid(np.arange(w, dtype=np.float64), np.arange(h, dtype=np.float64))
    for yaw, pitch in specs:
        cam = Camera(f, w / 2, h / 2, float(fish.C[2]), pitch, yaw, 0.0, float(fish.C[0]), float(fish.C[1]))
        dw = (np.stack([uu, vv, np.ones_like(uu)], -1) @ cam.Kinv.T) @ cam.R.T
        px, _ = kb_dir_to_px(fish, dw @ fish.R_cw)
        out.append({"cam": cam, "yaw": yaw, "pitch": pitch, "mapx": px[..., 0].astype(np.float32), "mapy": px[..., 1].astype(np.float32)})
    return out
def lens_K_for(W, H):
    """intrinsics of the frames the geometry sees (after any undistortion), or None if no lens profile is set."""
    if not CFG.lens: return None
    K = np.asarray(CFG.lens["K"], float)
    if CFG.lens.get("model") == "fisheye":
        return cv2.fisheye.estimateNewCameraMatrixForUndistortRectify(K, np.asarray(CFG.lens["D"], float).reshape(4, 1), (W, H), np.eye(3),
                                                                     balance=CFG.lens.get("balance", 0.0))
    return K


def undistorter(W, H):
    """frame → undistorted frame. CFG.lens = {'K', 'dist': [k1,k2,p1,p2,k3]} for ordinary lenses, or
    {'model': 'fisheye', 'K', 'D': [k1..k4], 'balance': 0..1} for fisheyes (≤ ~180°; wider: use fisheye_views)."""
    if not CFG.lens: return lambda f: f
    K = np.asarray(CFG.lens["K"], float); nK = lens_K_for(W, H)
    if CFG.lens.get("model") == "fisheye":
        m1, m2 = cv2.fisheye.initUndistortRectifyMap(K, np.asarray(CFG.lens["D"], float).reshape(4, 1), np.eye(3), nK, (W, H), cv2.CV_16SC2)
    else:
        m1, m2 = cv2.initUndistortRectifyMap(K, np.asarray(CFG.lens["dist"], float), None, nK, (W, H), cv2.CV_16SC2)
    return lambda f: cv2.remap(f, m1, m2, cv2.INTER_LINEAR)

# ─────────────────────────────── §4 CGA hazard zones ─────────────────────────────
_ep, _em = CGA.mv(ep=1), CGA.mv(em=1)
E0, EINF = 0.5 * (_em - _ep), _em + _ep
def cga_up(x): return CGA.mv(e1=x[0], e2=x[1], e3=x[2]) + 0.5 * float(np.dot(x, x)) * EINF + E0
def cga_sphere(c, r): return cga_up(c) - 0.5 * r * r * EINF
def cga_plane(n, d): return CGA.mv(e1=n[0], e2=n[1], e3=n[2]) + d * EINF
# vectorised 5-tuple (x,y,z,a,b) = x + a e∞ + b e₀ ; dot = cdot of the Lean file
def up5(X):
    X = np.atleast_2d(X); return np.concatenate([X, 0.5 * (X ** 2).sum(1, keepdims=True), np.ones((len(X), 1))], 1)
def sphere5(c, r): s = up5(np.asarray(c, float))[0]; s[3] -= 0.5 * r * r; return s
def plane5(n, d): n = np.asarray(n, float); n = n / np.linalg.norm(n); return np.array([*n, d, 0.0])
def cdot5(P, S):                                   # (N,5)·(K,5) → (N,K): one 5D dot per pair
    return P[:, :3] @ S[:, :3].T - (P[:, 3:4] * S[None, :, 4] + P[:, 4:5] * S[None, :, 3])

@dataclass
class Zone:
    name: str; kind: str; params: dict
    def vec5(self, t):
        if self.kind == "sphere":
            c = self.params["center"](t) if callable(self.params["center"]) else self.params["center"]
            return sphere5(c, self.params["r"])
        if self.kind == "halfspace": return plane5(self.params["n"], self.params["d"])
        return None

def polygon_inside(poly, pts):
    """convex floor polygon (CCW). Each edge = 2D-PGA join; sign of line·point (proved: side_of_line)."""
    poly = np.asarray(poly, float); pts = np.atleast_2d(pts)[:, :2]; inside = np.ones(len(pts), bool)
    for a, b in zip(poly, np.roll(poly, -1, 0)):
        inside &= ((b[0] - a[0]) * (pts[:, 1] - a[1]) - (b[1] - a[1]) * (pts[:, 0] - a[0])) >= 0
    return inside

def zone_hits(zones, body_pts, t):
    """body_pts (M,3) for one person → {zone: clearance_m (<0 = inside)}"""
    out = {}
    P = up5(body_pts)
    for z in zones:
        if z.kind == "polygon":
            ins = polygon_inside(z.params["poly"], body_pts[:1])[0]; out[z.name] = -0.01 if ins else 1.0
            continue
        s = z.vec5(t); v = cdot5(P, s[None])[:, 0].max()       # max over body points
        if z.kind == "sphere":
            r = z.params["r"]; out[z.name] = math.sqrt(max(r * r - 2 * v, 0.0)) - r   # d − r from ½(r²−d²)
        else:
            out[z.name] = -v                                    # n·x − δ > 0 means past the plane
    return out

def default_zones():
    return [
        Zone("pump_skid_keepout", "polygon", {"poly": [(-6, 8), (-2, 8), (-2, 11), (-6, 11)]}),
        Zone("crane_hook", "sphere", {"center": (3.0, 12.0, 1.0), "r": 2.0}),
        Zone("suspended_load", "sphere",
             {"center": lambda t: (-4 + 8 * (0.5 + 0.5 * math.sin(0.1 * t)), 14.0, 1.2), "r": 1.8}),
        Zone("rig_edge", "halfspace", {"n": (0, 1, 0), "d": 16.5}),
    ]

# ─────────────────────────────── §5 motors, tracker, fall rule ───────────────────
# SE(2) motors in Cl(2,0,1) acting on ground-plane points (bivectors)
_P0 = pga2_point(0, 0)
def m_translator(dx, dy):
    # T = 1 ∓ (dx/2) e01 ∓ (dy/2) e02 ; signs fixed once below for this basis ordering, then asserted
    return PGA2.mv(s=1.0) + _TSIGN[0] * 0.5 * dx * PGA2.mv(e01=1) + _TSIGN[1] * 0.5 * dy * PGA2.mv(e02=1)
def m_rotor(phi): return math.cos(phi / 2) * PGA2.mv(s=1.0) + _RSIGN * math.sin(phi / 2) * PGA2.mv(e12=1)
def m_apply(M, x, y): return pga2_coords(PGA2.sand(M, pga2_point(x, y)))
def m_log(M):
    M = M / math.hypot(M[0], M @ _F12)
    if M[0] < 0: M = -M          # ±M is the same rigid motion (double cover): take the short way round.
                                 # Without this, a track whose accumulated heading passes ±180° gets its
                                 # correction negated and is pushed AWAY from its measurement (found by the Rust port).
    s, a = M[0], M @ _F12
    th = math.atan2(a, s); B = M.copy(); B[0] = 0
    return B if abs(a) < 1e-9 else B * (th / a)
def m_exp(B):
    a = B @ _F12; out = B.copy()
    if abs(a) < 1e-12: out[0] = 1.0; return out
    out *= math.sin(a) / a; out[0] = math.cos(a); return out
def m_pose(M):
    p = m_apply(M, 0, 0); q = m_apply(M, 1, 0)
    return p, math.atan2(q[1] - p[1], q[0] - p[0])

# fix sign conventions once, empirically, then assert
_TSIGN, _RSIGN = [1, 1], 1
for sx in (1, -1):
    for sy in (1, -1):
        _TSIGN = [sx, sy]
        if np.allclose(m_apply(m_translator(2.0, -3.0), 1.0, 1.0), [3.0, -2.0]): break
    else: continue
    break
for _r in (1, -1):
    _RSIGN = _r
    if np.allclose(m_apply(m_rotor(math.pi / 2), 1.0, 0.0), [0.0, 1.0]): break

def rotor_angle_deg(a, b):
    """Cl(3,0) rotor R=(1+ba)/|1+ba| taking unit a to unit b; angle = 2·acos⟨R⟩₀."""
    a = np.asarray(a, float) / np.linalg.norm(a); b = np.asarray(b, float) / np.linalg.norm(b)
    R = CL3.gp(CL3.mv(e1=b[0], e2=b[1], e3=b[2]), CL3.mv(e1=a[0], e2=a[1], e3=a[2])); R[0] += 1
    nr = math.sqrt(CL3.gp(R, CL3.rev(R))[0])
    return math.degrees(2 * math.acos(min(1.0, R[0] / nr))) if nr > 1e-9 else 180.0
_E3 = (0, 0, 1)

BODY_LAT, BODY_SAG = 0.25, 0.17          # half-width, half-depth of the body cuboid (m)
def body_cuboid(origin_xy, H, tilt, yaw):
    """feet at origin; body axis tilted by `tilt` toward heading `yaw` (tilt=0 standing, π/2 lying)."""
    c = np.array([[sx * BODY_LAT, sy * BODY_SAG, z] for sx in (-1, 1) for sy in (-1, 1) for z in (0, H)])
    ct, st = math.cos(tilt), math.sin(tilt)
    c = c @ np.array([[1, 0, 0], [0, ct, st], [0, -st, ct]]).T
    cy_, sy_ = math.cos(yaw), math.sin(yaw)
    c = c @ np.array([[cy_, -sy_, 0], [sy_, cy_, 0], [0, 0, 1]]).T
    c[:, 2] = np.maximum(c[:, 2], 0.0)
    return c + np.array([origin_xy[0], origin_xy[1], 0.0])
def _iou(a, b):
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    i = ix * iy; return i / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - i + 1e-9)
POSE_HYPS = [(0.0, 0.0, H) for H in (1.6, 1.75, 1.9)] + \
            [(math.radians(t), math.radians(y), 1.75) for t in (45, 90) for y in range(0, 360, 45)]

_HYP_LOCAL = np.stack([body_cuboid((0.0, 0.0), H, t, y) for t, y, H in POSE_HYPS])     # (K,8,3)

def fit_pose(cam, box, foot):
    """Single-view 3D pose by analysis-by-synthesis, vectorised over all K hypotheses: render each body
    through the camera, slide it on the ground until its box bottom lands on the observed one, best IoU wins."""
    K = len(POSE_HYPS); o = np.repeat(np.asarray(foot, float)[None, :2], K, 0)
    def render(o):
        uv = cam.project((_HYP_LOCAL + np.c_[o, np.zeros(K)][:, None, :]).reshape(-1, 3)).reshape(K, 8, 2)
        return np.concatenate([uv.min(1), uv.max(1)], 1)
    for _ in range(3):
        bb = render(o)
        d = cam.R @ (cam.Kinv @ np.stack([(bb[:, 0] + bb[:, 2]) / 2, bb[:, 3], np.ones(K)]))   # rays (3,K)
        ok = d[2] < -1e-9
        g = cam.C[:2, None] + (-cam.C[2] / np.where(ok, d[2], -1.0)) * d[:2]                  # ray ∧ ground
        o = o + np.where(ok[:, None], foot[None, :2] - g.T, 0.0)
    bb = render(o)
    ix = np.clip(np.minimum(bb[:, 2], box[2]) - np.maximum(bb[:, 0], box[0]), 0, None)
    iy = np.clip(np.minimum(bb[:, 3], box[3]) - np.maximum(bb[:, 1], box[1]), 0, None)
    inter = ix * iy; iou = inter / ((bb[:, 2] - bb[:, 0]) * (bb[:, 3] - bb[:, 1]) + (box[2] - box[0]) * (box[3] - box[1]) - inter + 1e-9)
    k = int(iou.argmax()); cub = _HYP_LOCAL[k] + np.r_[o[k], 0.0]
    axis = cub[[1, 3, 5, 7]].mean(0) - cub[[0, 2, 4, 6]].mean(0)          # head-end minus feet-end
    return {"iou": float(iou[k]), "origin": o[k], "tilt": rotor_angle_deg(_E3, axis),
            "top_z": float(cub[:, 2].max()), "H": POSE_HYPS[k][2]}

class Track:
    _next = 1
    def __init__(self, xy, t):
        self.id = Track._next; Track._next += 1
        self.M = PGA2.gp(m_translator(*xy), m_rotor(0.0)); self.B = np.zeros(PGA2.N); self.v = np.zeros(2)
        self.t = t; self.hits = 1; self.hist = []        # (t, head_z, tilt, zone_inside)
        self.fall_latched = False; self.in_zone_since = {}; self.zone_fired = set()
        self.state = {}
    def predict(self, t):
        return PGA2.gp(m_exp(self.B * (t - self.t)), self.M)
    def update(self, xy, t):
        dt = max(t - self.t, 1e-3); Mp = self.predict(t)
        _, hd = m_pose(Mp)
        # a foot point observes position only → measurement motor shares the predicted heading, so the
        # geodesic error log(Mz·~Mp) is a pure translation and the blend cannot swing the track along an arc
        Mz = PGA2.gp(m_translator(*xy), m_rotor(hd))
        err = m_log(PGA2.gp(Mz, PGA2.rev(Mp)))
        Mnew = PGA2.gp(m_exp(CFG.blend * err), Mp)
        p_new, _ = m_pose(Mnew); p_old, _ = m_pose(self.M)
        vel = (p_new - p_old) / dt
        if np.linalg.norm(vel) > 0.3:                      # heading follows velocity, ≤ 20° per update
            d = (math.atan2(vel[1], vel[0]) - hd + math.pi) % (2 * math.pi) - math.pi
            hd += max(-math.radians(20), min(math.radians(20), d))
        Mnew = PGA2.gp(m_translator(*p_new), m_rotor(hd))
        # constant-velocity motor: translational generator only (people turn unpredictably; an angular
        # term would also feed back through the heading and spin the prediction)
        self.v = CFG.vel_alpha * vel + (1 - CFG.vel_alpha) * self.v
        self.B = m_log(m_translator(*self.v))
        self.M, self.t = Mnew, t; self.hits += 1
    @property
    def xy(self): return m_pose(self.M)[0]

class Tracker:
    def __init__(self): self.tracks = []
    def step(self, meas_xy, t):
        assign = [-1] * len(meas_xy)
        if self.tracks and meas_xy:
            pred = np.array([m_pose(tr.predict(t))[0] for tr in self.tracks])
            cost = np.linalg.norm(pred[:, None, :] - np.asarray(meas_xy)[None, :, :], axis=2)
            r, c = linear_sum_assignment(cost)
            for i, j in zip(r, c):
                if cost[i, j] < CFG.gate_m: self.tracks[i].update(meas_xy[j], t); assign[j] = self.tracks[i].id
        for j, a in enumerate(assign):
            if a < 0:
                tr = Track(meas_xy[j], t); self.tracks.append(tr); assign[j] = tr.id
        self.tracks = [tr for tr in self.tracks if t - tr.t <= CFG.max_age_s]
        return assign
    def get(self, tid): return next(tr for tr in self.tracks if tr.id == tid)

def emit(ev):
    EVENT_LOG.append(ev); return ev

def _recent_fall_near(t, xy, radius=2.0, window=12.0):
    return any(e["type"] == "fall" and t - e["t"] <= window and math.dist(e["xy"], xy) < radius
               for e in reversed(EVENT_LOG[-200:]))

def update_rules(tr, t, head_z, tilt, hits):
    """fall: was standing within 3 s, then 'down' sustained ≥ fall_sustain_s. intrusion: inside ≥ sustain."""
    events = []
    down = head_z < CFG.fall_head_z and tilt > CFG.fall_tilt_deg
    tr.hist.append((t, head_z, tilt, down)); tr.hist = [h for h in tr.hist if t - h[0] <= 6.0]
    stood = any(h[1] > CFG.stand_head_z for h in tr.hist if t - h[0] <= 3.0 + CFG.fall_sustain_s)
    down_for = 0.0
    for h in reversed(tr.hist):
        if not h[3]: break
        down_for = t - h[0]
    if down and down_for >= CFG.fall_sustain_s and stood and not tr.fall_latched:
        tr.fall_latched = True
        xy = [round(float(c), 2) for c in tr.xy]
        if not _recent_fall_near(t, xy):
            events.append(emit({"t": round(t, 2), "type": "fall", "track": tr.id, "xy": xy,
                                "head_z_m": round(head_z, 2), "tilt_deg": round(tilt, 1)}))
    if tr.fall_latched and sum(1 for h in tr.hist if t - h[0] <= 1.0 and h[1] > CFG.stand_head_z) >= 3:
        tr.fall_latched = False
    for zn, clr in hits.items():
        if clr < 0:
            tr.in_zone_since.setdefault(zn, t)
            if t - tr.in_zone_since[zn] >= CFG.intrusion_sustain_s and zn not in tr.zone_fired:
                tr.zone_fired.add(zn); events.append(emit({"t": round(t, 2), "type": "intrusion", "zone": zn,
                                                           "track": tr.id, "clearance_m": round(clr, 2)}))
        else:
            tr.in_zone_since.pop(zn, None); tr.zone_fired.discard(zn)
    tr.state = {"down": down, "fallen": tr.fall_latched, "head_z": head_z, "tilt": tilt,
                "inside": {k: v < 0 for k, v in hits.items()}}
    return events

def box_geometry(cam, b):
    """one person box → world feet (PGA ray ∧ deck), head height, fitted 3D pose; None if it can't be on the deck."""
    x1, y1, x2, y2 = b; uc = (x1 + x2) / 2
    ft = ground_point(cam, uc, y2)
    if ft is None or (ft - cam.C) @ cam.R[:, 2] <= 0: return None
    hz = head_height(cam, uc, y1, ft)
    pose = fit_pose(cam, np.asarray(b, float), ft)
    if pose["tilt"] > 1.0: hz = pose["top_z"]              # vertical-plane head height only valid when upright
    return {"box": b, "foot": np.array([*pose["origin"], 0.0]), "head_z": hz, "tilt": pose["tilt"], "pose_iou": pose["iou"]}

def finish_recs(valid, tracker, zones, t):
    """tracking, zone clearances and rules for per-person records (from one camera or merged views)."""
    ids = tracker.step([r["foot"][:2] for r in valid], t)
    for r, tid in zip(valid, ids):
        tr = tracker.get(tid); r["track"] = tid
        body = np.array([[*r["foot"][:2], 0.1], [*r["foot"][:2], 0.9], [*r["foot"][:2], max(r["head_z"], 0.2)]])
        r["zones"] = zone_hits(zones, body, t)
        r["events"] = update_rules(tr, t, r["head_z"], r["tilt"], r["zones"])
        r["state"] = tr.state
    return valid

def analyse_boxes(cam, tracker, zones, boxes, t):
    """boxes (x1,y1,x2,y2) of persons → per-box dict with world geometry, zone clearance, track state."""
    recs = [box_geometry(cam, b) for b in boxes]
    valid = [r for r in recs if r is not None]
    finish_recs(valid, tracker, zones, t)
    return recs

# ─────────────────────────────── self-tests: GA ≡ closed form ────────────────────
def _selftest():
    rng = np.random.default_rng(0)
    for _ in range(20):                                         # PGA point round-trip
        x = rng.normal(size=3) * 5; assert np.allclose(pga_coords(pga_point(*x)), x)
        p = rng.normal(size=2) * 5; assert np.allclose(pga2_coords(pga2_point(*p)), p)
    cam = Camera(CFG.focal_px, CFG.img_w / 2, CFG.img_h / 2, CFG.cam_height_m, CFG.pitch_deg, 20.0)
    for _ in range(20):                                         # PGA meet ≡ closed form ≡ ground truth
        X = np.array([rng.uniform(-5, 5), rng.uniform(6, 15), 0.0]); u, v = cam.project(X)[0]
        g1 = ground_point_pga(cam, u, v); g2 = ray_plane_fast(cam.C, cam.ray(u, v), np.array([0, 0, 1.0]), 0)
        assert np.allclose(g1, X, atol=1e-6) and np.allclose(g2, X, atol=1e-6)
        H = rng.uniform(1.5, 1.9); ut, vt = cam.project(X + [0, 0, H])[0]
        assert abs(head_height(cam, ut, vt, X) - H) < 1e-6
    for _ in range(20):                                         # CGA engine ≡ cdot5 (Lean `cdot`)
        x, y = rng.normal(size=3), rng.normal(size=3); r = rng.uniform(.5, 3); n = rng.normal(size=3); n /= np.linalg.norm(n)
        d = rng.normal()
        assert np.isclose(CGA.scalar(cga_up(x), cga_up(y)), -0.5 * np.sum((x - y) ** 2))
        assert np.isclose(CGA.scalar(cga_up(x), cga_sphere(y, r)), cdot5(up5(x), sphere5(y, r)[None])[0, 0])
        assert np.isclose(CGA.scalar(cga_up(x), cga_plane(n, d)), x @ n - d)
        assert np.isclose(cdot5(up5(x), plane5(n, d)[None])[0, 0], x @ n - d)
    for _ in range(20):                                         # motors: exp∘log = id, composition
        M = PGA2.gp(m_translator(*rng.normal(size=2)), m_rotor(rng.uniform(-3, 3)))
        assert np.allclose(m_exp(m_log(M)), M / math.hypot(M[0], M @ _F12), atol=1e-9)
    assert np.allclose(m_apply(m_translator(2.0, -3.0), 1, 1), [3, -2])
    assert np.allclose(m_apply(m_rotor(math.pi / 2), 1, 0), [0, 1], atol=1e-9)
    assert abs(rotor_angle_deg((0, 0, 1), (1, 0, 0)) - 90) < 1e-6
    T = Tracker(); Track._next = 1                               # 3 crossing walkers → exactly 3 identities
    pos = np.array([[0., 6], [3, 8], [-2, 10]]); vel = np.array([[1, .2], [-.5, .5], [.3, -.8]])
    for k in range(60): ids = T.step([p + v * k * .1 + rng.normal(0, .05, 2) for p, v in zip(pos, vel)], k * .1)
    assert ids == [1, 2, 3] and Track._next == 4, ids
    lying = cam.project(body_cuboid((1.0, 10.0), 1.75, math.pi / 2, 0.0))   # fall along the line of sight
    fp = fit_pose(cam, np.r_[lying.min(0), lying.max(0)], ground_point(cam, lying[:, 0].mean(), lying[:, 1].max()))
    assert fp["tilt"] > 60 and fp["iou"] > 0.95, fp
    # vanishing-point auto-calibration on a synthetic camera
    segs_x = [tuple(map(tuple, cam.project(np.array([[-3, y, 0], [3, y, 0]])))) for y in (7, 10, 13)]
    segs_y = [tuple(map(tuple, cam.project(np.array([[x, 6, 0], [x, 15, 0]])))) for x in (-3, 0, 3)]
    f_est, pitch_est = calibrate_from_vanishing_points(vanishing_point(segs_x), vanishing_point(segs_y), cam.cx, cam.cy)
    assert abs(f_est - cam.f) < 1e-3 and abs(pitch_est - cam.pitch) < 1e-3, (f_est, pitch_est)
    # point calibration: rolled, offset camera; 8 deck points with 0.5 px click noise
    true = Camera(1100.0, 960.0, 540.0, 9.0, 30.0, 20.0, roll=3.0, x0=2.0, y0=-1.0)
    W_ = [(-6, 8), (6, 8), (-8, 16), (8, 16), (0, 11), (-4, 22), (5, 25), (1, 6)]
    wpts = np.array([[true.x0 + x * math.cos(math.radians(-20)) - y * math.sin(math.radians(-20)),
                      true.y0 + x * math.sin(math.radians(-20)) + y * math.cos(math.radians(-20))] for x, y in W_])
    ipts = true.project(np.c_[wpts, np.zeros(len(wpts))]) + rng.normal(0, 0.5, (len(wpts), 2))
    cal = calibrate_from_points(ipts, wpts, (1920, 1080))
    assert abs(cal["f_px"] / 1100 - 1) < 0.03 and abs(cal["height_m"] - 9) < 0.25 and abs(cal["roll_deg"] - 3) < 0.5, cal
    assert abs(cal["pitch_deg"] - 30) < 0.5 and abs(cal["yaw_deg"] - 20) < 0.5 and cal["rms_m"] < 0.08, cal
    no_roll = Camera(cal["f_px"], 960, 540, cal["height_m"], cal["pitch_deg"], cal["yaw_deg"], 0.0, *cal["position_m"])
    nr = np.sqrt(np.mean([np.sum((ray_plane_fast(no_roll.C, no_roll.ray(u, v), np.array([0, 0, 1.0]), 0)[:2] - w) ** 2) for (u, v), w in zip(ipts, wpts)]))
    # lens distortion: barrel lens, points distorted by OpenCV's own model
    Kt = np.array([[1100, 0, 960], [0, 1100, 540], [0, 0, 1.0]]); dist = np.array([-0.28, 0.09, 0, 0, 0])
    Rwc = true.R.T; rvec, _ = cv2.Rodrigues(Rwc); tvec = -Rwc @ true.C
    dpts = cv2.projectPoints(np.c_[wpts, np.zeros(len(wpts))].astype(np.float64), rvec, tvec, Kt, dist)[0].reshape(-1, 2) + rng.normal(0, 0.5, (len(wpts), 2))
    with_lens = calibrate_from_points(dpts, wpts, (1920, 1080), K=Kt, dist=dist)
    no_lens = calibrate_from_points(dpts, wpts, (1920, 1080))
    # score on held-out check points spread over the frame (the fit absorbs distortion at its own points)
    gx_, gy_ = np.meshgrid(np.linspace(-9, 9, 7), np.linspace(6, 26, 6)); ho = np.c_[gx_.ravel(), gy_.ravel()]
    ho = np.array([[true.x0 + x * math.cos(math.radians(-20)) - y * math.sin(math.radians(-20)),
                    true.y0 + x * math.sin(math.radians(-20)) + y * math.cos(math.radians(-20))] for x, y in ho])
    hd = cv2.projectPoints(np.c_[ho, np.zeros(len(ho))].astype(np.float64), rvec, tvec, Kt, dist)[0].reshape(-1, 2)
    keep_ = (hd[:, 0] > 0) & (hd[:, 0] < 1920) & (hd[:, 1] > 0) & (hd[:, 1] < 1080); ho, hd = ho[keep_], hd[keep_]
    def deck_rms(cam_, px_):
        g_ = np.array([ray_plane_fast(cam_.C, cam_.ray(u, v), np.array([0, 0, 1.0]), 0)[:2] for u, v in px_]); return float(np.sqrt(np.mean(np.sum((g_ - ho) ** 2, 1))))
    hu = cv2.undistortPoints(hd.reshape(-1, 1, 2), Kt, dist, P=Kt).reshape(-1, 2)
    lens_ok, lens_bad = deck_rms(with_lens["camera"], hu), deck_rms(no_lens["camera"], hd)
    assert lens_ok < 0.1 and lens_bad > 2 * lens_ok, (lens_ok, lens_bad)
    # fisheye: 185° equidistant-ish lens looking straight down from 10 m
    fw_ = make_fisheye([[892, 0, 1440], [0, 892, 1440], [0, 0, 1]], [0.02, -0.01, 0.002, -0.0005],
                       np.stack([[1, 0, 0], [0, -1, 0], [0, 0, -1.0]], 1), [0, 0, 10.0], (2880, 2880))   # columns: right, down, forward
    gp_ = np.array([[r * math.cos(a), r * math.sin(a), 0] for r in (1, 5, 10, 20, 30) for a in np.linspace(0, 2 * math.pi, 9)[:-1]])
    px_, th_ = kb_project(fw_, gp_)
    rv_, tv_ = cv2.Rodrigues(fw_.R_cw.T)[0], -fw_.R_cw.T @ fw_.C; ok_ = th_ < math.radians(85)
    cvp = cv2.fisheye.projectPoints(gp_[ok_].reshape(-1, 1, 3), rv_, tv_, fw_.K, fw_.D.reshape(4, 1))[0].reshape(-1, 2)
    assert np.max(np.abs(cvp - px_[ok_])) < 0.05, "fisheye model disagrees with cv2.fisheye"
    assert np.nanmax(np.linalg.norm(fisheye_ground(fw_, px_) - gp_[:, :2], axis=1)) < 1e-6, "fisheye ray round trip"
    vw_ = fisheye_views(fw_, [(0.0, 60.0), (90.0, 60.0)], size=(640, 480), hfov=70)
    for v in vw_:                                       # a deck point seen in a view lands where the fisheye says it is
        q = gp_[(v["cam"].project(gp_)[:, 0] > 5) & (v["cam"].project(gp_)[:, 0] < 635) & (v["cam"].project(gp_)[:, 1] > 5) & (v["cam"].project(gp_)[:, 1] < 475) & (((gp_ - v["cam"].C) @ v["cam"].R[:, 2]) > 0)]
        pv = v["cam"].project(q); fx_ = cv2.remap(v["mapx"], pv[:, 0].astype(np.float32).reshape(-1, 1), pv[:, 1].astype(np.float32).reshape(-1, 1), cv2.INTER_LINEAR).ravel()
        fy_ = cv2.remap(v["mapy"], pv[:, 0].astype(np.float32).reshape(-1, 1), pv[:, 1].astype(np.float32).reshape(-1, 1), cv2.INTER_LINEAR).ravel()
        assert len(q) and np.max(np.hypot(fx_ - kb_project(fw_, q)[0][:, 0], fy_ - kb_project(fw_, q)[0][:, 1])) < 0.5, "virtual view remap"
    ring = {}
    for r in (1, 10, 20, 30):                           # 0.5 px click noise → deck error at the centre vs the rim
        pts = np.array([[r * math.cos(a), r * math.sin(a), 0] for a in np.linspace(0, 2 * math.pi, 64)])
        noisy = kb_project(fw_, pts)[0] + rng.normal(0, 0.5, (64, 2))
        ring[f"{r} m ({math.degrees(math.atan2(r, 10)):.0f}° off axis)"] = round(float(np.median(np.linalg.norm(fisheye_ground(fw_, noisy) - pts[:, :2], axis=1))), 3)
    return {"f_est": f_est, "pitch_est": pitch_est, "fisheye_deck_error_m": ring,
            "point_calib": {k: round(cal[k], 3) for k in ("f_px", "height_m", "pitch_deg", "yaw_deg", "roll_deg", "rms_px", "rms_m")},
            "deck_rms_m": {"full_model": round(cal["rms_m"], 3), "ignoring_3deg_roll": round(float(nr), 3),
                           f"barrel_lens_undistorted ({len(ho)} check pts)": round(lens_ok, 3), "barrel_lens_ignored": round(lens_bad, 3)}}
REPORT["selftest"] = _selftest()
print(f"[§2-5] self-tests passed: PGA≡closed form, CGA≡cdot, motors, rotors, VP calibration "
      f"f={REPORT['selftest']['f_est']:.1f} pitch={REPORT['selftest']['pitch_est']:.2f}")
print(f"      point calibration (8 clicked deck points, 0.5 px noise, rolled + offset camera): {REPORT['selftest']['point_calib']}")
print(f"      deck error RMS, metres: {REPORT['selftest']['deck_rms_m']}")
print(f"      fisheye (185°, 10 m, looking down): model ≡ cv2.fisheye, virtual views ≡ fisheye; deck error from 0.5 px clicks: {REPORT['selftest']['fisheye_deck_error_m']}")

# ─────────────────────────────── §6 YOLOX-Nano detector (CPU) ────────────────────
def _download(url, path):
    path = Path(path)
    if not path.exists():
        r = requests.get(url, timeout=120); r.raise_for_status(); path.write_bytes(r.content)
    return path

def _letterbox(img, size):
    h, w = img.shape[:2]; r = min(size / h, size / w)
    out = np.full((size, size, 3), 114, np.uint8)
    out[: int(h * r), : int(w * r)] = cv2.resize(img, (int(w * r), int(h * r)), interpolation=cv2.INTER_LINEAR)
    return np.ascontiguousarray(out.transpose(2, 0, 1)[None].astype(np.float32)), r

def _to3(gray_or_bgr):
    return cv2.cvtColor(gray_or_bgr, cv2.COLOR_GRAY2BGR) if gray_or_bgr.ndim == 2 else gray_or_bgr

class YOLOX:
    def __init__(self, path, size=416):
        so = ort.SessionOptions(); so.intra_op_num_threads = CFG.ort_threads
        so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        self.sess = ort.InferenceSession(str(path), so, providers=["CPUExecutionProvider"])
        self.inp = self.sess.get_inputs()[0].name; self.size = size
        grids, strides = [], []
        for s in (8, 16, 32):
            n = size // s; xv, yv = np.meshgrid(np.arange(n), np.arange(n))
            grids.append(np.stack((xv, yv), 2).reshape(-1, 2)); strides.append(np.full((n * n, 1), s))
        self.grids, self.strides = np.concatenate(grids), np.concatenate(strides)
    def __call__(self, frame, classes=(0,)):
        img = _to3(frame); x, r = _letterbox(img, self.size)
        out = self.sess.run(None, {self.inp: x})[0][0]
        xy = (out[:, :2] + self.grids) * self.strides; wh = np.exp(out[:, 2:4]) * self.strides
        scores = out[:, 4:5] * out[:, 5:]; cls = scores.argmax(1); conf = scores.max(1)
        keep = np.isin(cls, classes) & (conf > CFG.det_conf)
        xy, wh, conf, cls = xy[keep], wh[keep], conf[keep], cls[keep]
        boxes = np.concatenate([xy - wh / 2, xy + wh / 2], 1) / r
        if not len(boxes): return np.zeros((0, 4)), np.zeros(0), np.zeros(0, int)
        idx = cv2.dnn.NMSBoxes(np.c_[boxes[:, :2], boxes[:, 2:] - boxes[:, :2]].tolist(), conf.tolist(),
                               CFG.det_conf, CFG.det_nms)
        idx = np.array(idx).reshape(-1)
        return boxes[idx], conf[idx], cls[idx]

def quantize_detector(fp32_path, calib_frames, size):
    from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantFormat, QuantType
    from onnxruntime.quantization.shape_inference import quant_pre_process
    pre = BASE / "yolox_nano.pre.onnx"; q = BASE / "yolox_nano.int8.onnx"
    import onnx
    from onnx import version_converter
    m = onnx.load(str(fp32_path))
    if max(o.version for o in m.opset_import if o.domain in ("", "ai.onnx")) < 13:       # per-channel DQ needs ≥13
        m = version_converter.convert_version(m, 13); fp32_path = BASE / "yolox_nano.op13.onnx"; onnx.save(m, str(fp32_path))
    quant_pre_process(str(fp32_path), str(pre))
    name = ort.InferenceSession(str(fp32_path), providers=["CPUExecutionProvider"]).get_inputs()[0].name
    class Reader(CalibrationDataReader):
        def __init__(self): self.it = iter([{name: _letterbox(_to3(f), size)[0]} for f in calib_frames])
        def get_next(self): return next(self.it, None)
    g = onnx.load(str(pre)).graph
    consumers = {}
    for nd in g.node:
        for i in nd.input: consumers.setdefault(i, []).append(nd)
    def feeds_head(nd):                      # conv whose output goes straight into the output Concat/Sigmoid
        return any(c.op_type in ("Concat", "Sigmoid") for o in nd.output for c in consumers.get(o, []))
    exclude = [nd.name for nd in g.node if nd.op_type == "Conv" and feeds_head(nd)]
    quantize_static(str(pre), str(q), Reader(), quant_format=QuantFormat.QDQ, per_channel=True,
                    weight_type=QuantType.QInt8, activation_type=QuantType.QUInt8,
                    op_types_to_quantize=["Conv"], nodes_to_exclude=exclude)
    return q

def _calib_frames(img, n=24):
    rng = np.random.default_rng(1); g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img; out = []
    for _ in range(n):                                        # brightness / contrast / flip / crop jitter
        a = rng.uniform(0.5, 1.4); b = rng.uniform(-40, 40)
        f = np.clip(g.astype(np.float32) * a + b, 0, 255).astype(np.uint8)
        if rng.random() < .5: f = f[:, ::-1]
        h, w = f.shape; y0, x0 = rng.integers(0, h // 6), rng.integers(0, w // 6)
        out.append(np.ascontiguousarray(f[y0:, x0:]))
    return out

def _bench(det, frame, n=20):
    det(frame); t0 = time.perf_counter()
    for _ in range(n): det(frame)
    return (time.perf_counter() - t0) / n * 1000

demo_path = _download(CFG.demo_image_url, BASE / "demo.jpg")
demo_img = cv2.imread(str(demo_path)); demo_gray = cv2.cvtColor(demo_img, cv2.COLOR_BGR2GRAY)
fp32 = Path(CFG.detector_onnx) if CFG.detector_onnx else _download(CFG.detector_url, BASE / "yolox_nano.onnx")
det_fp32 = YOLOX(fp32, CFG.det_size)
REPORT["det_ms_fp32"] = _bench(det_fp32, demo_gray)
DET = det_fp32
if CFG.quantize_int8:
    try:
        calib = _calib_frames(demo_gray)
        if CFG.source and Path(CFG.source).exists():
            cap = cv2.VideoCapture(CFG.source); fr = []
            while len(fr) < 48:
                ok, f = cap.read()
                if not ok: break
                fr.append(cv2.cvtColor(f, cv2.COLOR_BGR2GRAY))
            calib = fr[::2] or calib
        det_int8 = YOLOX(quantize_detector(fp32, calib, CFG.det_size), CFG.det_size)
        REPORT["det_ms_int8"] = _bench(det_int8, demo_gray)
        b32, _, _ = det_fp32(demo_gray); b8, _, _ = det_int8(demo_gray)
        REPORT["det_persons_fp32_vs_int8"] = (len(b32), len(b8))
        same = abs(len(b8) - len(b32)) <= max(1, len(b32) // 4)
        DET = det_int8 if same and REPORT["det_ms_int8"] < 0.95 * REPORT["det_ms_fp32"] else det_fp32   # int8 only if it pays on THIS cpu (needs VNNI/AMX)
    except Exception as e:
        REPORT["det_int8_error"] = repr(e)
print(f"[§6] detector fp32 {REPORT['det_ms_fp32']:.1f} ms/frame"
      + (f", int8 {REPORT['det_ms_int8']:.1f} ms/frame, persons fp32/int8 = {REPORT['det_persons_fp32_vs_int8']}"
         if "det_ms_int8" in REPORT else f" (int8 skipped: {REPORT.get('det_int8_error','off')})")
      + f" · using {'int8' if DET is not det_fp32 else 'fp32'}")

# ─────────────────────────────── §7 PPE head classifier: Clifford conv + SAM ─────
def _cayley(cl):
    N = cl.N; M = torch.zeros(N, N, N)
    for i in range(N):
        for j in range(N):
            M[i, j, cl.gp_idx[i, j]] = float(cl.gp_sgn[i, j])
    return M

class CliffordConv2d(nn.Module):
    """Cl(2,0) multivector channels [1,e1,e2,e12]. out = Σ W ⊙geo x. Weight (Cout,Cin,4,k,k) → folded real conv
    (4Cout,4Cin,k,k): same FLOPs at inference, 4× fewer parameters, geometric coupling of intensity & gradient."""
    def __init__(self, cin, cout, k=3, stride=1, padding=1):
        super().__init__()
        self.cin, self.cout, self.k, self.stride, self.padding = cin, cout, k, stride, padding
        self.weight = nn.Parameter(torch.randn(cout, cin, 4, k, k) * math.sqrt(2.0 / (cin * 4 * k * k)))
        self.bias = nn.Parameter(torch.zeros(cout))          # scalar part only
        self.register_buffer("M", _cayley(CL2))
    def real_weight(self):
        W = torch.einsum("ocixy,ijk->okcjxy", self.weight, self.M)
        return W.reshape(self.cout * 4, self.cin * 4, self.k, self.k)
    def real_bias(self):
        b = torch.zeros(self.cout, 4, device=self.bias.device); b[:, 0] = self.bias; return b.reshape(-1)
    def forward(self, x): return F.conv2d(x, self.real_weight(), self.real_bias(), self.stride, self.padding)
    def folded(self):
        c = nn.Conv2d(self.cin * 4, self.cout * 4, self.k, self.stride, self.padding)
        with torch.no_grad(): c.weight.copy_(self.real_weight()); c.bias.copy_(self.real_bias())
        return c

class MVGate(nn.Module):
    """x · σ(⟨x⟩₀): scales each multivector by a gate on its scalar part (keeps blade structure)."""
    def forward(self, x):
        B, C, H, W = x.shape; m = x.view(B, C // 4, 4, H, W)
        return (m * torch.sigmoid(m[:, :, :1])).view(B, C, H, W)

class Lift(nn.Module):
    """gray image → multivector field [I, ∂x I, ∂y I, 0]: the gradient is literally a vector."""
    def __init__(self):
        super().__init__()
        k = torch.tensor([[-1., 0, 1], [-2, 0, 2], [-1, 0, 1]]) / 8
        self.register_buffer("kx", k[None, None]); self.register_buffer("ky", k.t()[None, None])
    def forward(self, x):
        return torch.cat([x, F.conv2d(x, self.kx, padding=1), F.conv2d(x, self.ky, padding=1), torch.zeros_like(x)], 1)

class Readout(nn.Module):
    """invariant-style readout: per multivector channel [⟨x⟩₀, |⟨x⟩₁|, ⟨x⟩₂] then linear."""
    def __init__(self, c, n_cls):
        super().__init__(); self.fc = nn.Linear(3 * c, n_cls)
    def forward(self, x):
        B, C = x.shape[:2]; m = x.mean((2, 3)).view(B, C // 4, 4)
        f = torch.cat([m[..., 0], torch.sqrt(m[..., 1] ** 2 + m[..., 2] ** 2 + 1e-6), m[..., 3]], 1)
        return self.fc(f)

class Standardize(nn.Module):
    """per-crop zero-mean / unit-variance: removes CCTV gain & exposure differences (inside the ONNX graph)."""
    def forward(self, x):
        m = x.mean((1, 2, 3), keepdim=True); return (x - m) / (x.std((1, 2, 3), keepdim=True) + 1e-3)

def make_ppe_net(width=(6, 12, 16)):
    a, b, c = width
    return nn.Sequential(Standardize(), Lift(),
                         CliffordConv2d(1, a), MVGate(), nn.AvgPool2d(2),
                         CliffordConv2d(a, b), MVGate(), nn.AvgPool2d(2),
                         CliffordConv2d(b, c), MVGate(), nn.AvgPool2d(2),
                         CliffordConv2d(c, c), MVGate(),
                         Readout(c, 2))

def fold_for_export(net):
    return nn.Sequential(*[m.folded() if isinstance(m, CliffordConv2d) else m for m in net])

class SAM(torch.optim.Optimizer):
    """Sharpness-Aware Minimization (Foret et al. 2021): ascend to w+ε within ρ-ball, descend with that gradient."""
    def __init__(self, params, base_optimizer, rho=0.05, adaptive=False, **kw):
        super().__init__(params, dict(rho=rho, adaptive=adaptive, **kw))
        self.base_optimizer = base_optimizer(self.param_groups, **kw); self.param_groups = self.base_optimizer.param_groups
    @torch.no_grad()
    def first_step(self):
        gn = torch.norm(torch.stack([((torch.abs(p) if g["adaptive"] else 1.0) * p.grad).norm(2)
                                     for g in self.param_groups for p in g["params"] if p.grad is not None]), 2)
        for g in self.param_groups:
            scale = g["rho"] / (gn + 1e-12)
            for p in g["params"]:
                if p.grad is None: continue
                self.state[p]["old_p"] = p.data.clone()
                p.add_((torch.pow(p, 2) if g["adaptive"] else 1.0) * p.grad * scale)
    @torch.no_grad()
    def second_step(self):
        for g in self.param_groups:
            for p in g["params"]:
                if p.grad is not None: p.data = self.state[p]["old_p"]
        self.base_optimizer.step()

def synth_heads(n, size, rng):
    """Synthetic gray head crops — a pipeline smoke test, NOT a substitute for real site data."""
    X = np.zeros((n, size, size), np.float32); y = rng.integers(0, 2, n)
    for i in range(n):
        img = np.full((size, size), rng.uniform(40, 200), np.float32)
        img += cv2.GaussianBlur(rng.normal(0, 25, (size, size)).astype(np.float32), (0, 0), 3)
        cx, cy = size / 2 + rng.normal(0, 2), size * 0.58 + rng.normal(0, 2); r = size * rng.uniform(.2, .27)
        skin = rng.uniform(70, 190)
        cv2.ellipse(img, (int(cx), int(cy)), (int(r), int(r * 1.25)), 0, 0, 360, skin, -1)
        if y[i]:   # helmet: wider smooth dome + brim, bright or dark shell, specular highlight
            shell = rng.choice([rng.uniform(190, 250), rng.uniform(25, 70)])
            cv2.ellipse(img, (int(cx), int(cy - r * .35)), (int(r * 1.22), int(r * 1.05)), 0, 180, 360, shell, -1)
            cv2.line(img, (int(cx - r * 1.45), int(cy - r * .3)), (int(cx + r * 1.45), int(cy - r * .3)), shell * .9, 2)
            cv2.circle(img, (int(cx - r * .4), int(cy - r * .9)), max(1, int(r * .18)), min(255, shell + 40), -1)
        else:      # hard negatives: dark hair cap / beanie (no brim, tighter fit)
            if rng.random() < .7:
                cv2.ellipse(img, (int(cx), int(cy - r * .3)), (int(r * 1.02), int(r * .95)), 0, 180, 360,
                            rng.uniform(20, 120), -1)
        M = cv2.getRotationMatrix2D((size / 2, size / 2), rng.uniform(-25, 25), rng.uniform(.85, 1.15))
        img = cv2.warpAffine(img, M, (size, size), borderMode=cv2.BORDER_REFLECT)
        img = cv2.GaussianBlur(img, (0, 0), rng.uniform(.3, 1.4)) + rng.normal(0, rng.uniform(3, 14), img.shape)
        X[i] = np.clip(img, 0, 255) / 255.0
    return X[:, None], y

def voc_to_crops(voc_root, out_dir, label_map={"hat": "helmet", "person": "no_helmet"}, size=48):
    """Pascal-VOC head boxes (e.g. SHWD: 'hat' / 'person') → ImageFolder crops for CFG.ppe_dir."""
    import xml.etree.ElementTree as ET
    voc_root, out_dir = Path(voc_root), Path(out_dir)
    for xml in (voc_root / "Annotations").glob("*.xml"):
        root = ET.parse(xml).getroot(); img = cv2.imread(str(voc_root / "JPEGImages" / root.findtext("filename")), 0)
        if img is None: continue
        for k, ob in enumerate(root.iter("object")):
            lab = label_map.get(ob.findtext("name"))
            if not lab: continue
            bb = [int(float(ob.find("bndbox").findtext(t))) for t in ("xmin", "ymin", "xmax", "ymax")]
            crop = img[max(bb[1], 0):bb[3], max(bb[0], 0):bb[2]]
            if crop.size < 64: continue
            (out_dir / lab).mkdir(parents=True, exist_ok=True)
            cv2.imwrite(str(out_dir / lab / f"{xml.stem}_{k}.png"), cv2.resize(crop, (size, size)))

def load_ppe(size, rng):
    d = Path(CFG.ppe_dir) if CFG.ppe_dir else None
    if d and (d / "helmet").exists() and (d / "no_helmet").exists():
        X, y = [], []
        for lab, sub in ((1, "helmet"), (0, "no_helmet")):
            for p in sorted((d / sub).glob("*")):
                im = cv2.imread(str(p), 0)
                if im is not None: X.append(cv2.resize(im, (size, size)).astype(np.float32) / 255); y.append(lab)
        return np.array(X)[:, None], np.array(y), "real:" + str(d)
    X, y = synth_heads(5000, size, rng); return X, y, "synthetic (smoke test)"

def train_ppe():
    rng = np.random.default_rng(SEED)
    X, y, src = load_ppe(CFG.ppe_size, rng)
    idx = rng.permutation(len(X)); n_te = len(X) // 5
    te, tr = idx[:n_te], idx[n_te:]
    Xtr, ytr = torch.tensor(X[tr]), torch.tensor(y[tr]); Xte, yte = torch.tensor(X[te]), y[te]
    net = make_ppe_net()
    opt = SAM(net.parameters(), torch.optim.SGD, rho=CFG.sam_rho, lr=0.05, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt.base_optimizer, CFG.ppe_epochs)
    lossf = nn.CrossEntropyLoss(label_smoothing=0.05)
    for ep in range(CFG.ppe_epochs):
        net.train(); perm = torch.randperm(len(Xtr))
        for k in range(0, len(Xtr), 128):
            b = perm[k:k + 128]; xb, yb = Xtr[b], ytr[b]
            if random.random() < .5: xb = torch.flip(xb, dims=[3])
            lossf(net(xb), yb).backward(); opt.first_step(); opt.zero_grad()
            lossf(net(xb), yb).backward(); opt.second_step(); opt.zero_grad()
        sched.step()
    net.eval()
    with torch.no_grad(): pred = net(Xte).argmax(1).numpy()
    mcc = matthews_corrcoef(yte, pred)
    folded = fold_for_export(net).eval()
    with torch.no_grad(): assert torch.allclose(folded(Xte[:64]), net(Xte[:64]), atol=1e-4)
    onnx_path = BASE / "ppe_clifford_folded.onnx"
    kw = dict(input_names=["crop"], output_names=["logits"], dynamic_axes={"crop": {0: "n"}, "logits": {0: "n"}},
              opset_version=17)
    try: torch.onnx.export(folded, Xte[:1], str(onnx_path), dynamo=False, **kw)
    except TypeError: torch.onnx.export(folded, Xte[:1], str(onnx_path), **kw)
    sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    o = sess.run(None, {"crop": Xte[:64].numpy()})[0]
    with torch.no_grad(): assert np.allclose(o, net(Xte[:64]).numpy(), atol=1e-3)
    t0 = time.perf_counter()
    for _ in range(50): sess.run(None, {"crop": Xte[:8].numpy()})
    n_cliff = sum(p.numel() for m in net if isinstance(m, CliffordConv2d) for p in (m.weight,))
    n_real = sum(m.weight.numel() for m in folded if isinstance(m, nn.Conv2d))
    return sess, {"source": src, "n_test": int(n_te), "mcc": float(mcc),
                  "confusion[[tn,fp],[fn,tp]]": confusion_matrix(yte, pred).tolist(),
                  "conv_params_clifford": int(n_cliff), "conv_params_folded_real": int(n_real),
                  "onnx_ms_per_8_crops": (time.perf_counter() - t0) / 50 * 1000, "onnx": str(onnx_path)}

PPE_SESS, REPORT["ppe"] = train_ppe()
print(f"[§7] PPE Clifford-conv + SAM on {REPORT['ppe']['source']}: MCC={REPORT['ppe']['mcc']:.3f} "
      f"(n={REPORT['ppe']['n_test']}), conv params {REPORT['ppe']['conv_params_clifford']} → folded "
      f"{REPORT['ppe']['conv_params_folded_real']}, ONNX {REPORT['ppe']['onnx_ms_per_8_crops']:.2f} ms / 8 crops")

def head_crop(gray, box, size):
    x1, y1, x2, y2 = box; h = y2 - y1; s = max(0.24 * h, 8); cx = (x1 + x2) / 2; top = y1 - 0.03 * h
    xa, ya = int(max(cx - s / 2, 0)), int(max(top, 0)); xb, yb = int(min(cx + s / 2, gray.shape[1])), int(min(top + s, gray.shape[0]))
    if xb - xa < 4 or yb - ya < 4: return None
    return cv2.resize(gray[ya:yb, xa:xb], (size, size)).astype(np.float32) / 255

def ppe_classify(gray, boxes):
    crops = [head_crop(gray, b, CFG.ppe_size) for b in boxes]; ok = [i for i, c in enumerate(crops) if c is not None]
    out = [None] * len(boxes)
    if ok:
        logits = PPE_SESS.run(None, {"crop": np.stack([crops[i] for i in ok])[:, None]})[0]
        p = np.exp(logits - logits.max(1, keepdims=True)); p /= p.sum(1, keepdims=True)
        for i, pi in zip(ok, p[:, 1]): out[i] = float(pi)
    return out

# ─────────────────────────────── §7b Rule Studio engine ──────────────────────────
# Same rule ids, controlled-English and formal-logic templates as VG360 Rule Studio. Each rule compiles to a
# predicate over the geometry state of one person in one frame; the engine debounces, prioritises, writes a
# SHA-256 hash-chained evidence ledger, and exports a file Rule Studio can load.
import hashlib
TIER = {"P1": {"floor": 0.85, "cooldown_s": 0.0}, "P2": {"floor": 0.75, "cooldown_s": 30.0},
        "P3": {"floor": 0.60, "cooldown_s": 300.0}, None: {"floor": 0.80, "cooldown_s": 30.0}}
ACTIVITY = {"suspended": True, "pipe_handling": True}   # live feeds in production (crane load cell, drilling control system)

@dataclass
class Rule:
    id: str; hz: str; pri: str; name: str; en: str; fol: str; params: dict; cond: object; secs_key: str = ""
    def secs(self):
        if not self.secs_key: return 0.0
        v = float(self.params[self.secs_key]); return v / CFG.sim_fps if self.secs_key == "frames" else v
    def fill(self, s, hl=False): return re.sub(r"\{(\w+)\}", lambda m: str(self.params[m.group(1)]), s)

RULES = [
    Rule("lift", "H01", "P1", "Person under a suspended load",
         "If a load is suspended by {crane} and a person is inside {zone} for more than {secs} seconds, then raise a lifting alert.",
         "∀p ∀t. Person(p) ∧ Suspended(load({crane}), t) ∧ InZoneFor(p, {zone}, t, {secs}s) → Alert(lifting, p, t)",
         {"crane": "Crane 1", "zone": "suspended_load", "secs": 3},
         lambda s, c, P: c["suspended"] and s["clear"].get(P["zone"], 1) < 0, "secs"),
    Rule("red", "H02", "P1", "Red zone during pipe handling",
         "If a person enters the red zone on {zone} while {act} is active, then alert the driller.",
         "∀p ∀t. Person(p) ∧ Active({act}, t) ∧ In(p, redzone({zone}), t) → Alert(red_zone, p, t)",
         {"zone": "pump_skid_keepout", "act": "pipe handling"},
         lambda s, c, P: c["pipe_handling"] and s["clear"].get(P["zone"], 1) < 0),
    Rule("edge", "", None, "Unguarded deck edge",
         "If a person is within {m} metres of an unguarded deck edge in {zone}, then raise an edge alert.",
         "∀p ∀t. Person(p) ∧ In(p, {zone}, t) ∧ ∃e. UnguardedEdge(e) ∧ Dist(p, e, t) ≤ {m} → Alert(edge, p, t)",
         {"m": 1.0, "zone": "rig_edge"},
         lambda s, c, P: s["clear"].get(P["zone"], 99) < P["m"]),
    Rule("fall", "H06", "P2", "Slip, trip or person down",
         "If a person in {zone} falls and stays down for {secs} seconds, or a fluid spill is detected on a walkway, then raise an alert and record a near-miss.",
         "∀t. (∃p. Person(p) ∧ In(p, {zone}, t) ∧ For(Down(p), t, {secs}s)) ∨ Spill(walkway({zone}), t) → Alert(fall_or_spill, {zone}, t)",
         {"zone": "site", "secs": 2},
         lambda s, c, P: bool(s["down"]), "secs"),
    Rule("ppe", "H03", "P1", "Missing PPE",
         "If a person in {zone} is not wearing {ppe} for {frames} consecutive frames, then notify the supervisor and log a PPE alert.",
         "∀p ∀t. Person(p) ∧ In(p, {zone}, t) ∧ ForFrames(¬Wearing(p, {ppe}), t, {frames}) → Alert(ppe, p, t)",
         {"zone": "site", "ppe": "a hard hat", "frames": 15},
         lambda s, c, P: s.get("p_helmet") is not None and s["p_helmet"] < 0.5, "frames"),
]

class Ledger:
    """append-only evidence ledger: hash = SHA-256 of the canonical JSON [time, rule, who, clip, prev]."""
    def __init__(self): self.entries = []
    @staticmethod
    def digest(e): return hashlib.sha256(json.dumps([e["time"], e["rule"], e["who"], e["clip"], e["prev"]],
                                                    separators=(",", ":"), ensure_ascii=False).encode()).hexdigest()
    def append(self, time_s, rule, who, clip):
        e = {"time": f"{time_s:.1f}", "rule": rule, "who": who, "clip": clip,
             "prev": self.entries[-1]["hash"] if self.entries else "0" * 64}
        e["hash"] = self.digest(e); self.entries.append(e); return e
    def verify(self):
        prev = "0" * 64
        for e in self.entries:
            if e["prev"] != prev or self.digest(e) != e["hash"]: return False
            prev = e["hash"]
        return True

class RuleEngine:
    """per (subject, rule): run length of the condition; alert when it reaches the rule's duration.
    emitted = alert ∧ (P1 ∨ ¬recent) — exactly `emitted` in the Lean file (debounce never drops a P1)."""
    def __init__(self, rules, ledger=None, gap_s=None):
        self.rules, self.ledger = rules, ledger; self.run = {}; self.last = {}; self.active = {}; self.off = {}
        self.gap = CFG.rule_gap_s if gap_s is None else gap_s   # a dropout shorter than this doesn't reset the timer
    def held(self, key, rule):                          # condition has held for the rule's duration
        return self.run.get((key, rule.id), 0.0) >= max(rule.secs(), 1e-9)
    def step(self, key, state, ctx, t, dt, clip="", emit_events=True):
        fired = []
        for r in self.rules:
            k = (key, r.id); on = bool(r.cond(state, ctx, r.params))
            if on: self.off[k] = 0.0; self.run[k] = self.run.get(k, 0.0) + dt
            else:
                self.off[k] = self.off.get(k, 0.0) + dt
                if self.off[k] > self.gap: self.run[k] = 0.0
            alert = self.held(key, r) and not self.active.get(k, False)       # rising edge = one alert per episode
            self.active[k] = self.held(key, r)
            recent = t - self.last.get(k, -1e9) < TIER[r.pri]["cooldown_s"]
            if alert and (r.pri == "P1" or not recent):
                self.last[k] = t; fired.append(r.id)
                if emit_events:
                    ev = emit({"t": round(t, 2), "type": "rule", "rule": r.id, "hz": r.hz, "pri": r.pri,
                               "name": r.name, "track": key})
                    if self.ledger is not None: ev["hash"] = self.ledger.append(t, r.id, f"Track {key}", clip)["hash"]
        return fired

def rule_state(rec, p_helmet=None):
    return {"clear": rec["zones"], "down": rec["state"]["down"], "p_helmet": p_helmet}

def export_rule_studio(rule_cms, ledger, path, source):
    out = {"vg360_notebook_export": 1, "generated": time.strftime("%Y-%m-%dT%H:%M:%S"), "source": source,
           "rules": {}, "ledger": ledger.entries, "ledger_verified": ledger.verify()}
    for r in RULES:
        cm = rule_cms.get(r.id)
        out["rules"][r.id] = {"hz": r.hz, "pri": r.pri, "name": r.name, "en": r.fill(r.en), "fol": r.fill(r.fol),
                              "floor": TIER[r.pri]["floor"], "cm": cm, "validated_on": rule_cms.get(r.id + ":src", source),
                              "mcc": None if cm is None else float(matthews_corrcoef(
                                  [1] * cm[0] + [0] * cm[1] + [1] * cm[2] + [0] * cm[3],
                                  [1] * cm[0] + [1] * cm[1] + [0] * cm[2] + [0] * cm[3]))}
    Path(path).write_text(json.dumps(out, indent=1, ensure_ascii=False)); return out

# ─────────────────────────────── §8 simulated site → per-event MCC ───────────────
def simulate_and_score():
    """Renders people as projected cuboids (walk / fall / get up) → boxes with pixel noise and dropouts,
    runs §3–§5 exactly as on real detections, and scores against 3D ground truth."""
    rng = np.random.default_rng(SEED + 1)
    cam = Camera(CFG.focal_px, CFG.img_w / 2, CFG.img_h / 2, CFG.cam_height_m, CFG.pitch_deg, CFG.yaw_deg)
    zones = default_zones(); tracker = Tracker(); Track._next = 1
    def visible(p, H=1.8):
        uv = cam.project(np.array([[p[0], p[1], 0], [p[0], p[1], H]]))
        return np.all((uv[:, 0] > 20) & (uv[:, 0] < CFG.img_w - 20) & (uv[:, 1] > 5) & (uv[:, 1] < CFG.img_h - 5))
    def rand_pt():
        while True:
            p = np.array([rng.uniform(-9, 9), rng.uniform(5, 17.5)])
            if visible(p): return p
    people = []
    for i in range(CFG.sim_people):
        falls = rng.random() < 0.45
        people.append({"pos": rand_pt(), "goal": rand_pt(), "H": rng.uniform(1.6, 1.85), "v": rng.uniform(.6, 1.4),
                       "heading": 0.0, "t_fall": rng.uniform(8, CFG.sim_seconds - 15) if falls else None,
                       "fell": falls, "fall_dir": rng.uniform(-math.pi, math.pi)})
    cuboid = body_cuboid                       # same body model as fit_pose (optimistic: see notes)
    dt = 1 / CFG.sim_fps; rows = []; fired_falls = {}; frame_lat = []
    ENG_P = RuleEngine(RULES, SIM_LEDGER); ENG_G = RuleEngine(RULES, gap_s=0.0)   # predicted vs strict ground truth
    sim_ctx = lambda t: {"suspended": (t % 40) < 26, "pipe_handling": (t % 30) < 18}   # simulated activity feeds
    for k in range(int(CFG.sim_seconds * CFG.sim_fps)):
        t = k * dt; boxes, gt = [], []
        for pid, P in enumerate(people):
            tilt = 0.0; down = False
            if P["t_fall"] is not None and t >= P["t_fall"]:
                s = (t - P["t_fall"]); up_at = 10.0
                if s < 0.8: tilt = (math.pi / 2) * s / 0.8
                elif s < up_at: tilt = math.pi / 2; down = True
                elif s < up_at + 1.0: tilt = (math.pi / 2) * (1 - (s - up_at))
                else: P["t_fall"] = None
            if tilt == 0.0:
                d = P["goal"] - P["pos"]
                if np.linalg.norm(d) < 0.3: P["goal"] = rand_pt()
                else:
                    P["heading"] = math.atan2(d[1], d[0])
                    nxt = P["pos"] + d / np.linalg.norm(d) * P["v"] * dt
                    P["pos"] = nxt if visible(nxt) else P["pos"]
            yaw = P["fall_dir"] if tilt > 0 else 0.0
            corners = cuboid(P["pos"], P["H"], tilt, yaw)
            uv = cam.project(corners)
            if (uv[:, 1] > CFG.img_h).any() or (uv[:, 0] < 0).any() or (uv[:, 0] > CFG.img_w).any(): continue
            if rng.random() < 0.05: continue                                  # detector miss
            b = np.r_[uv.min(0), uv.max(0)] + rng.normal(0, 2.0, 4)
            top = corners[:, 2].max()
            body = np.array([[*P["pos"], 0.1], [*P["pos"], 0.9], [*P["pos"], max(top, 0.2)]])
            clr = zone_hits(zones, body, t); ins = {z: v < 0 for z, v in clr.items()}
            boxes.append(b); gt.append((pid, down, ins, clr))
        t0 = time.perf_counter(); recs = analyse_boxes(cam, tracker, zones, boxes, t)
        frame_lat.append((time.perf_counter() - t0) * 1000 / max(len(boxes), 1))
        ctx = sim_ctx(t)
        for r, (pid, down, ins, clr) in zip(recs, gt):
            if r is None: continue
            ENG_P.step(r["track"], rule_state(r), ctx, t, dt, clip=f"sim-{int(t * CFG.sim_fps)}")
            ENG_G.step(f"gt{pid}", {"clear": clr, "down": down, "p_helmet": None}, ctx, t, dt, emit_events=False)
            rows.append({"pid": pid, "t": t, "gt_down": down, "pred_down": r["state"]["down"],
                         **{f"rg_{x.id}": ENG_G.held(f"gt{pid}", x) for x in RULES},
                         **{f"rp_{x.id}": ENG_P.held(r["track"], x) for x in RULES},
                         **{f"gt_{z}": ins[z] for z in ins}, **{f"pred_{z}": r["state"]["inside"][z] for z in ins}})
            for e in r["events"]:
                if e["type"] == "fall": fired_falls.setdefault(pid, []).append(t)
    res = {}
    for z in [z.name for z in zones]:
        g = np.array([r[f"gt_{z}"] for r in rows]); p = np.array([r[f"pred_{z}"] for r in rows])
        res[f"intrusion:{z}"] = {"MCC": float(matthews_corrcoef(g, p)), "positives": int(g.sum()), "n": len(g)}
    g = np.array([r["gt_down"] for r in rows]); p = np.array([r["pred_down"] for r in rows])
    res["person_down(frame)"] = {"MCC": float(matthews_corrcoef(g, p)), "positives": int(g.sum()), "n": len(g)}
    ge = np.array([P["fell"] for P in people]); pe = np.array([pid in fired_falls for pid in range(len(people))])
    res["fall_event(person)"] = {"MCC": float(matthews_corrcoef(ge, pe)), "positives": int(ge.sum()), "n": len(ge),
                                 "confusion[[tn,fp],[fn,tp]]": confusion_matrix(ge, pe, labels=[0, 1]).tolist()}
    res["geometry_ms_per_person"] = float(np.median(frame_lat))
    res["identities"] = {"people": len(people), "tracks_created": Track._next - 1}
    for x in RULES:
        if x.id == "ppe": continue                                  # no imagery in the simulation
        g = np.array([r[f"rg_{x.id}"] for r in rows]); p = np.array([r[f"rp_{x.id}"] for r in rows])
        tn, fp, fn, tp = confusion_matrix(g, p, labels=[0, 1]).ravel()
        RULE_CMS[x.id] = [int(tp), int(fp), int(fn), int(tn)]
        res[f"rule:{x.id} ({x.pri or '-'})"] = {"MCC": float(matthews_corrcoef(g, p)), "floor": TIER[x.pri]["floor"],
                                                "positives": int(g.sum()), "n": len(g)}
    return res

EVENT_LOG.clear(); SIM_LEDGER = Ledger(); RULE_CMS = {}
REPORT["sim"] = simulate_and_score()
print("[§8] simulated site — MCC per safety event (geometry stages, detector replaced by noisy projected boxes):")
for k, v in REPORT["sim"].items():
    print(f"      {k:32s} {v if not isinstance(v, dict) else {kk: (round(vv, 3) if isinstance(vv, float) else vv) for kk, vv in v.items()}}")
SIM_EVENTS = list(EVENT_LOG)
pt, pfp, pfn, ptn = (lambda c: (c[1][1], c[0][1], c[1][0], c[0][0]))(REPORT["ppe"]["confusion[[tn,fp],[fn,tp]]"])
RULE_CMS["ppe"] = [pt, pfp, pfn, ptn]; RULE_CMS["ppe:src"] = "PPE crop test set: " + REPORT["ppe"]["source"]
REPORT["rule_studio"] = export_rule_studio(RULE_CMS, SIM_LEDGER, BASE / "rule_studio_export.json",
                                           "notebook simulated site (noisy projected boxes; bodies share the fall-fitter's shape)")
print(f"[§8b] Rule Studio: {len(SIM_LEDGER.entries)} alerts in a SHA-256 ledger, verified={SIM_LEDGER.verify()} → "
      f"{BASE/'rule_studio_export.json'} (load it with Rule Studio's 'Load notebook results')")
for x in RULES:
    c = REPORT["rule_studio"]["rules"][x.id]
    m = "n/a" if c["mcc"] is None else f"{c['mcc']:.3f}"
    print(f"      {x.hz or '—':4s} {x.pri or '—':3s} {x.name:34s} MCC {m:>6s}  floor {c['floor']:.2f}  {'meets' if c['mcc'] is not None and c['mcc'] >= c['floor'] else 'BELOW'}")
t_ = SIM_LEDGER.entries[0]["who"] if SIM_LEDGER.entries else None
if SIM_LEDGER.entries:                                          # tamper test: edit an old entry → chain must break
    keep = SIM_LEDGER.entries[0]["who"]; SIM_LEDGER.entries[0]["who"] = "Track 999"
    assert not SIM_LEDGER.verify(); SIM_LEDGER.entries[0]["who"] = keep; assert SIM_LEDGER.verify()
    print("      tamper test: editing entry 0 breaks the chain ✓, restoring it verifies again ✓")

# ─────────────────────────────── §9 run on real frames ───────────────────────────
def rig_zones():
    """drill floor seen from a camera high on the derrick / crane boom (metres, origin below the camera)."""
    return [
        Zone("well_centre", "sphere", {"center": (0.0, 14.0, 1.0), "r": 2.2}),           # rotary table / pipe handling
        Zone("pipe_deck_keepout", "polygon", {"poly": [(3, 12), (7, 12), (7, 18), (3, 18)]}),
        Zone("crane_load", "sphere", {"center": lambda t: (-3 + 6 * math.sin(0.03 * t), 20.0, 2.0), "r": 2.0}),
        Zone("deck_edge", "halfspace", {"n": (-1, 0, 0), "d": 8.0}),                      # x < -8 m is over the side
    ]
SCENES = {   # camera height (m), pitch down (deg), horizontal FOV (deg), zones
    "rig_elevated": dict(h=12.0, pitch=35.0, hfov=70.0, zones=rig_zones,      # ≤ 40° down: steeper views shrink workers
                         rule_zones={"lift": "crane_load", "red": "well_centre", "edge": "deck_edge"}),
    "rig_demo":     dict(h=10.0, pitch=25.0, hfov=70.0, zones=rig_zones,      # camera of the synthetic oil-rig demo clip
                         rule_zones={"lift": "crane_load", "red": "well_centre", "edge": "deck_edge"}),
    "street_demo":  dict(h=CFG.cam_height_m, pitch=CFG.pitch_deg, hfov=2 * math.degrees(math.atan(CFG.img_w / 2 / CFG.focal_px)),
                         zones=default_zones),
}

def detect_tiled(gray, tiles=None, overlap=0.25):
    """full frame + tiles×tiles overlapping crops through the same 416 px detector, merged by NMS and by
    dropping boxes that sit mostly inside a stronger one (people cut by a tile edge)."""
    tiles = CFG.tiles if tiles is None else tiles
    B, S = [], []
    b, c, _ = DET(gray); B += list(b); S += list(c)
    if tiles and tiles > 1:
        H, W = gray.shape[:2]; th, tw = int(H / tiles * (1 + overlap)), int(W / tiles * (1 + overlap))
        for i in range(tiles):
            for j in range(tiles):
                y0 = min(int(i * H / tiles), H - th); x0 = min(int(j * W / tiles), W - tw)
                b, c, _ = DET(np.ascontiguousarray(gray[max(y0, 0):y0 + th, max(x0, 0):x0 + tw]))
                B += [bb + np.array([x0, y0, x0, y0]) for bb in b]; S += list(c)
    if not B: return np.zeros((0, 4)), np.zeros(0)
    B, S = np.array(B), np.array(S)
    idx = np.array(cv2.dnn.NMSBoxes(np.c_[B[:, :2], B[:, 2:] - B[:, :2]].tolist(), S.tolist(), CFG.det_conf, CFG.det_nms)).reshape(-1)
    B, S = B[idx], S[idx]; order = np.argsort(-S); keep = []
    for i in order:
        a = B[i]; area = (a[2] - a[0]) * (a[3] - a[1])
        inside = any(max(0, min(a[2], B[k][2]) - max(a[0], B[k][0])) * max(0, min(a[3], B[k][3]) - max(a[1], B[k][1])) > 0.8 * area
                     for k in keep)
        if not inside: keep.append(i)
    return B[keep], S[keep]

def scene_camera(name, W, H, h=None):
    sc = SCENES[name]
    if sc.get("calib"):                                   # measured deck points beat preset height/pitch/FOV
        cal = calibrate_from_points(sc["calib"]["image_pts"], sc["calib"]["world_pts"], (W, H), K=lens_K_for(W, H), dist=None)
        REPORT.setdefault("calibration", {})[name] = {k: v for k, v in cal.items() if k not in ("camera", "H")}
        print(f"[§9] {name} calibrated from {cal['n']} points: f={cal['f_px']:.0f}px h={cal['height_m']:.2f}m pitch={cal['pitch_deg']:.1f}° "
              f"roll={cal['roll_deg']:.1f}° · {cal['rms_px']:.2f} px / {cal['rms_m']:.3f} m RMS" + "".join(f"\n      ! {w}" for w in cal["warnings"]))
        return cal["camera"]
    f = (W / 2) / math.tan(math.radians(sc["hfov"] / 2))
    return Camera(f, W / 2, H / 2, h or sc["h"], sc["pitch"], CFG.yaw_deg)

def maybe_upload():
    if CFG.upload_photo and IN_COLAB:
        from google.colab import files
        up = files.upload()                                   # pick a rig photo or clip
        if up:
            name = next(iter(up)); p = BASE / name; p.write_bytes(up[name]); CFG.source = str(p)
            print(f"[§9] using uploaded {name} with scene preset '{CFG.scene}'")

def draw(frame, recs, ppe, zones, cam, t, show_zones=True):
    vis = _to3(frame.copy())
    def ground_poly(P, closed=True):                         # project ground points in front of the camera
        P = np.asarray(P, float); P = P[((P - cam.C) @ cam.R)[:, 2] > 0.5]
        if len(P) >= 2: cv2.polylines(vis, [cam.project(P).astype(np.int32)], closed, (0, 200, 255), 2)
    for z in (zones if show_zones else []):                   # zone footprints on the deck
        if z.kind == "polygon":
            ground_poly(np.c_[np.asarray(z.params["poly"]), np.zeros(len(z.params["poly"]))])
        elif z.kind == "sphere":
            c = z.params["center"](t) if callable(z.params["center"]) else z.params["center"]; r = z.params["r"]
            re = max(math.sqrt(max(r * r - (hb - c[2]) ** 2, 0)) for hb in (0.1, 0.9, 1.7))   # widest body-height slice
            a = np.linspace(0, 2 * math.pi, 48); ground_poly(np.c_[c[0] + re * np.cos(a), c[1] + re * np.sin(a), np.zeros(48)])
        else:
            n = np.asarray(z.params["n"], float); n /= np.linalg.norm(n); d = np.array([-n[1], n[0], 0.0])
            ground_poly([n * z.params["d"] + s_ * d for s_ in np.linspace(-40, 40, 81)], closed=False)
    for r, p in zip(recs, ppe):
        if r is None: continue
        x1, y1, x2, y2 = map(int, r["box"]); bad = any(r["state"]["inside"].values()) or r["state"]["fallen"]
        col = (0, 0, 255) if bad else (0, 255, 0)
        cv2.rectangle(vis, (x1, y1), (x2, y2), col, 2)
        lab = f"#{r['track']} ({r['foot'][0]:.1f},{r['foot'][1]:.1f})m h={r['head_z']:.2f}"
        if p is not None: lab += f" helmet={p:.2f}"
        cv2.putText(vis, lab, (x1, max(12, y1 - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.4, col, 1)
    return vis

# ─────────────────────────────── §9a oil-rig demo clip (synthetic composite) ───────
# A pipe deck rendered through the rig_elevated camera, with the real people from the street photo cut out
# (GrabCut) and stood on the deck at known positions, scaled to their true size at that range. Not real rig
# footage: it exercises detection, PGA metres, CGA zones and the Rule Studio engine with known ground truth.
def _render_deck(cam, W, H, rng):
    uu, vv = np.meshgrid(np.arange(W, dtype=np.float32), np.arange(H, dtype=np.float32))
    d = (np.stack([uu, vv, np.ones_like(uu)], -1) @ cam.Kinv.T.astype(np.float32)) @ cam.R.T.astype(np.float32)
    t = np.where(d[..., 2] < -1e-6, -cam.C[2] / np.minimum(d[..., 2], -1e-6), 1e3)
    gx, gy = cam.C[0] + t * d[..., 0], cam.C[1] + t * d[..., 1]
    plate = 118 + 10 * ((np.floor(gx / 2) + np.floor(gy / 2)) % 2)                       # 2 m deck plates
    seam = (np.minimum(np.abs(gx / 2 - np.round(gx / 2)), np.abs(gy / 2 - np.round(gy / 2))) * 2 < 0.04)
    deck = plate - 45 * seam + rng.normal(0, 6, (H, W))
    img = np.repeat(deck[..., None], 3, 2)
    far = (d[..., 2] >= -1e-6) | (t > 120)                                                      # beyond the deck: haze / sky
    img[far] = np.stack([215 - 40 * (vv / H), 200 - 40 * (vv / H), 185 - 40 * (vv / H)], -1)[far]
    sea = (gx < -8.0) & ~far
    w_ = 8 * np.sin(gy * 3 + gx); img[sea] = np.stack([104 + w_, 84 + w_, 58 + w_], -1)[sea]           # BGR blue-grey sea
    stripe = (gx >= -8.0) & (gx < -7.6)
    img[stripe] = np.where((((gx + gy) / 0.35).astype(int) % 2 == 0)[..., None], [20, 200, 235], [25, 25, 25])[stripe]
    r = np.hypot(gx - 0.0, gy - 14.0)                                                         # well centre red zone
    band = (r > 2.0) & (r < 2.3); img[band] = np.where(((np.arctan2(gy - 14, gx) * 8 / np.pi).astype(int) % 2 == 0)[..., None], [40, 40, 210], [235, 235, 235])[band]
    img[r < 0.8] = 40
    pz = ((gx > 3) & (gx < 7) & (gy > 12) & (gy < 18)) & ~((gx > 3.15) & (gx < 6.85) & (gy > 12.15) & (gy < 17.85))
    img[pz] = [20, 200, 235]                                                                  # pipe-deck keep-out paint
    return np.clip(img, 0, 255).astype(np.uint8)

def _draw_box(img, cam, lo, hi, base):
    c = np.array([[x, y, z] for x in (lo[0], hi[0]) for y in (lo[1], hi[1]) for z in (lo[2], hi[2])], float)
    faces = [([0, 1, 3, 2], [-1, 0, 0]), ([4, 5, 7, 6], [1, 0, 0]), ([0, 1, 5, 4], [0, -1, 0]), ([2, 3, 7, 6], [0, 1, 0]), ([1, 3, 7, 5], [0, 0, 1])]
    vis = [(np.linalg.norm(c[f].mean(0) - cam.C), f, n) for f, n in faces if np.dot(n, cam.C - c[f].mean(0)) > 0]
    for _, f, n in sorted(vis, reverse=True):
        shade = 1.15 if n[2] else (0.8 if n[0] else 0.65)
        cv2.fillConvexPoly(img, cam.project(c[f]).astype(np.int32), [int(min(255, v * shade)) for v in base], cv2.LINE_AA)
        cv2.polylines(img, [cam.project(c[f]).astype(np.int32)], True, (30, 30, 30), 1, cv2.LINE_AA)

def _cutouts(donor):
    g = cv2.cvtColor(donor, cv2.COLOR_BGR2GRAY); boxes, _, _ = det_fp32(g); outs = []
    for b in boxes:
        x1, y1, x2, y2 = [int(v) for v in b]; x1, y1 = max(x1, 0), max(y1, 0); x2, y2 = min(x2, donor.shape[1] - 1), min(y2, donor.shape[0] - 1)
        if (x2 - x1) < 20 or (y2 - y1) < 60: continue
        m = np.zeros(donor.shape[:2], np.uint8); bg, fg = np.zeros((1, 65), np.float64), np.zeros((1, 65), np.float64)
        cv2.grabCut(donor, m, (x1, y1, x2 - x1, y2 - y1), bg, fg, 4, cv2.GC_INIT_WITH_RECT)
        mask = np.where((m == cv2.GC_FGD) | (m == cv2.GC_PR_FGD), 255, 0).astype(np.uint8)[y1:y2, x1:x2]
        outs.append((donor[y1:y2, x1:x2].copy(), cv2.GaussianBlur(mask, (5, 5), 0)))
    return outs

def _paste(img, cam, crop, mask, xy, height=1.75):
    foot, head = cam.project(np.array([[xy[0], xy[1], 0.0], [xy[0], xy[1], height]]))
    hp = max(8, int(abs(foot[1] - head[1]))); s = hp / crop.shape[0]
    c = cv2.resize(crop, (max(4, int(crop.shape[1] * s)), hp)); m = cv2.resize(mask, (c.shape[1], hp)).astype(np.float32)[..., None] / 255
    x0, y0 = int(foot[0] - c.shape[1] / 2), int(foot[1] - hp)
    cv2.ellipse(img, (int(foot[0]), int(foot[1])), (max(3, c.shape[1] // 2), max(2, hp // 10)), 0, 0, 360, (60, 60, 60), -1, cv2.LINE_AA)
    ys, xs = slice(max(y0, 0), min(y0 + hp, img.shape[0])), slice(max(x0, 0), min(x0 + c.shape[1], img.shape[1]))
    cy, cx = slice(ys.start - y0, ys.stop - y0), slice(xs.start - x0, xs.stop - x0)
    if ys.stop > ys.start and xs.stop > xs.start:
        img[ys, xs] = (m[cy, cx] * c[cy, cx] + (1 - m[cy, cx]) * img[ys, xs]).astype(np.uint8)

RIG_DEMO_PEOPLE = [   # (start xy, velocity m/s) — what each one is doing
    ((0.6, 13.6), (0.0, 0.0)),     # standing in the well-centre red zone
    ((-3.4, 20.6), (0.05, 0.0)),   # under the suspended load
    ((-7.3, 10.0), (0.0, 0.3)),    # walking along the unguarded deck edge
    ((5.0, 9.5), (0.3, 0.2)),      # walking across open deck, safe
]
def make_rig_demo(path, frames=40, fps=10, W=1920, H=1080):   # 1080p, like a real site camera
    rng = np.random.default_rng(3); cam = scene_camera("rig_demo", W, H)
    bgimg = _render_deck(cam, W, H, rng)
    for i in range(5):                                                         # tubulars on the pipe deck
        for k in range(2): _draw_box(bgimg, cam, (3.5 + 0.6 * i + 0.3 * k, 12.6, 0.42 * k), (3.9 + 0.6 * i + 0.3 * k, 17.4, 0.42 * k + 0.4), (70, 90, 110))
    _draw_box(bgimg, cam, (11, 24, 0), (17, 31, 2.6), (120, 140, 120))           # mud tank
    _draw_box(bgimg, cam, (10, 8, 0), (14, 11, 3.0), (150, 150, 165))            # doghouse
    load_zone = next(z for z in rig_zones() if z.name == "crane_load")
    cuts = _cutouts(demo_img); writer = cv2.VideoWriter(str(path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H)); truth = []
    for f in range(frames):
        img = bgimg.copy(); t = f / fps; pos = []
        for j, ((x, y), (vx, vy)) in enumerate(RIG_DEMO_PEOPLE):
            p = (x + vx * t, y + vy * t); pos.append(p); crop, mask = cuts[j % len(cuts)]
            _paste(img, cam, crop if j < len(cuts) else crop[:, ::-1], mask if j < len(cuts) else mask[:, ::-1], p)
        lc = load_zone.params["center"](t)                                      # load follows the zone's own motion
        _draw_box(img, cam, (lc[0] - 1, lc[1] - 0.6, 2.2), (lc[0] + 1, lc[1] + 0.6, 3.1), (40, 110, 160))
        top = cam.project(np.array([[lc[0], lc[1], 3.1], [lc[0], lc[1], 11.5]])).astype(int)
        cv2.line(img, tuple(top[0]), tuple(top[1]), (20, 20, 20), 2, cv2.LINE_AA)
        img = cv2.GaussianBlur(img, (3, 3), 0); writer.write(img); truth.append(pos)
    writer.release(); return truth

LIVE_LEDGER = Ledger(); LIVE_ENGINE = RuleEngine(RULES, LIVE_LEDGER)
def run_source(source=None, cam=None, zones=None, out_name="annotated"):
    maybe_upload()
    global RIG_DEMO_TRUTH
    src = source or CFG.source
    if not src:                                                          # default: the synthetic oil-rig demo clip
        src = str(BASE / "rig_demo.mp4"); RIG_DEMO_TRUTH = make_rig_demo(src); scene = "rig_demo"; out_name = "rig_demo_annotated"
    else:
        RIG_DEMO_TRUTH = None; scene = CFG.scene
    saved_zone_params = {r.id: r.params.get("zone") for r in RULES}
    for r in RULES:                                                      # point each rule at this scene's zone names
        if r.id in SCENES[scene].get("rule_zones", {}): r.params["zone"] = SCENES[scene]["rule_zones"][r.id]
    is_img = Path(src).suffix.lower() in (".jpg", ".jpeg", ".png", ".bmp")
    frames = [cv2.imread(src)] if is_img else None
    cap = None if is_img else cv2.VideoCapture(src)
    first = frames[0] if is_img else cap.read()[1]
    H, W = first.shape[:2]
    zones = zones or SCENES[scene]["zones"](); tracker = Tracker(); Track._next = 1
    h_used = SCENES[scene]["h"]
    if cam is None:
        cam = scene_camera(scene, W, H)
        if CFG.auto_height and scene != "street_demo" and RIG_DEMO_TRUTH is None and not SCENES[scene].get("calib"):
            g0 = cv2.cvtColor(first, cv2.COLOR_BGR2GRAY) if first.ndim == 3 else first
            b0, _ = detect_tiled(g0)
            upright = [tuple(b) for b in b0 if (b[3] - b[1]) > 1.6 * (b[2] - b[0])]
            if len(upright) >= 2:
                kw = dict(f=cam.f, cx=cam.cx, cy=cam.cy, h=cam.h, pitch=cam.pitch, yaw=cam.yaw)
                h_used = float(np.clip(calibrate_height_from_people(kw, upright), 3.0, 80.0)); cam = scene_camera(scene, W, H, h_used)
    writer = None if is_img else cv2.VideoWriter(str(BASE / f"{out_name}.mp4"), cv2.VideoWriter_fourcc(*"mp4v"),
                                                 cap.get(cv2.CAP_PROP_FPS) or 10, (W, H))
    fps = (cap.get(cv2.CAP_PROP_FPS) if cap else 10) or 10; k = 0; frame = first; lat = []; per_frame = []
    undist = undistorter(W, H)                           # identity unless CFG.lens is set (calibration points use undistorted pixels)
    while frame is not None and k < CFG.max_frames:
        frame = undist(frame)
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) if frame.ndim == 3 else frame
        t0 = time.perf_counter()
        boxes, conf = detect_tiled(gray, tiles=0 if scene == "street_demo" else None)
        recs = analyse_boxes(cam, tracker, zones, [tuple(b) for b in boxes], k / fps)
        ppe = ppe_classify(gray, [tuple(b) for b in boxes])
        for r, p in zip(recs, ppe):
            if r is not None:
                LIVE_ENGINE.step(r["track"], rule_state(r, p), ACTIVITY, k / fps, 1 / fps, clip=f"{out_name}-{k}")
        for r, p in zip(recs, ppe):
            if r is not None and p is not None and p < 0.5 and r["head_z"] > CFG.stand_head_z:
                tr = tracker.get(r["track"])
                if not tr.state.get("ppe_fired"):
                    tr.state["ppe_fired"] = True
                    emit({"t": round(k / fps, 2), "type": "ppe_no_helmet", "track": r["track"], "p_helmet": round(p, 2)})
        lat.append((time.perf_counter() - t0) * 1000); per_frame.append(len(boxes))
        vis = draw(gray, recs, ppe, zones, cam, k / fps, show_zones=scene != "street_demo")
        cv2.imwrite(str(BASE / f"{out_name}.jpg"), vis); last_recs = recs; last_k = k
        if is_img: break
        writer.write(vis); ok, frame = cap.read(); frame = frame if ok else None; k += 1
    if writer: writer.release()
    for r in RULES: r.params["zone"] = saved_zone_params[r.id] if saved_zone_params[r.id] is not None else r.params.get("zone")
    err = None
    if RIG_DEMO_TRUTH is not None:                                       # metric accuracy against the known positions
        gt = np.array(RIG_DEMO_TRUTH[last_k]); est = [r["foot"][:2] for r in last_recs if r is not None]
        err = [round(float(np.min(np.linalg.norm(gt - e, axis=1))), 2) for e in est]
    return {"scene": scene, "camera_h_m": round(h_used, 2), "frames": max(k, 1), "persons_last": len(boxes),
            "ms_per_frame_end_to_end": float(np.median(lat)), "position_error_m": err,
            "workers_detected_per_frame": f"median {int(np.median(per_frame))}, min {min(per_frame)}, max {max(per_frame)}",
            "rule_alerts": [f"{e['rule']}@{e['time']}s" for e in LIVE_LEDGER.entries],
            "output": str(BASE / (f"{out_name}.jpg" if is_img else f"{out_name}.mp4")), "last_frame": str(BASE / f"{out_name}.jpg")}

EVENT_LOG.clear(); EVENT_LOG.extend(SIM_EVENTS)
REPORT["run"] = run_source()
print(f"[§9] {REPORT['run']}")
if REPORT["run"]["position_error_m"] is not None:
    print("      oil-rig demo: synthetic composite (rendered pipe deck + real people cut from the street photo) — "
          "position error per detected worker, metres:", REPORT["run"]["position_error_m"])
if IN_COLAB:
    from google.colab.patches import cv2_imshow
    cv2_imshow(cv2.imread(REPORT["run"]["last_frame"]))

# ─────────────────────────────── §9b fisheye cameras: virtual views, one tracker ─────
def fisheye_from_pinhole_maps(fish, cam, src_size, chunk=360):
    """remap tables: fisheye pixel → pixel of a pinhole camera with the same centre (to resample a pinhole clip)."""
    Wf, Hf = fish.size; mx = np.full((Hf, Wf), -1, np.float32); my = np.full((Hf, Wf), -1, np.float32)
    uu = np.arange(Wf, dtype=np.float64)
    for y0 in range(0, Hf, chunk):
        ys = np.arange(y0, min(y0 + chunk, Hf), dtype=np.float64); U, V = np.meshgrid(uu, ys)
        d = kb_ray(fish, np.c_[U.ravel(), V.ravel()]) @ cam.R                    # world ray → pinhole camera frame
        ok = d[:, 2] > 1e-3; u = np.where(ok, cam.f * d[:, 0] / np.where(ok, d[:, 2], 1) + cam.cx, -1); v = np.where(ok, cam.f * d[:, 1] / np.where(ok, d[:, 2], 1) + cam.cy, -1)
        mx[y0:y0 + len(ys)] = u.reshape(len(ys), Wf); my[y0:y0 + len(ys)] = v.reshape(len(ys), Wf)
    return mx, my

def run_fisheye(frames, fish, specs, zones, truth=None, fps=10, out_name="fisheye_annotated", hfov=60.0, view_size=(1280, 720)):
    """frames: iterable of fisheye BGR frames. Each frame → upright virtual views → detection (2×2 tiles per view) →
    per-view geometry → duplicates merged in metres across views → one tracker, zones, rules, ledger."""
    views = fisheye_views(fish, specs, size=view_size, hfov=hfov); tracker = Tracker(); Track._next = 1
    writer, errs, counts, lat = None, [], [], []
    for k, fr in enumerate(frames):
        t = k / fps; t0 = time.perf_counter(); g = cv2.cvtColor(fr, cv2.COLOR_BGR2GRAY) if fr.ndim == 3 else fr; cand = []
        for vi, v in enumerate(views):
            img = cv2.remap(g, v["mapx"], v["mapy"], cv2.INTER_LINEAR, borderValue=0)
            boxes, _ = detect_tiled(img, tiles=2)
            ppe = ppe_classify(img, [tuple(b) for b in boxes])
            for b, p in zip(boxes, ppe):
                r = box_geometry(v["cam"], tuple(b))
                if r is not None: r.update(view=vi, p_helmet=p, size=b[3] - b[1]); cand.append(r)
        kept = []
        for r in sorted(cand, key=lambda r: -r["size"]):                       # same worker seen in two views → keep one
            if all(np.linalg.norm(r["foot"][:2] - q["foot"][:2]) > 0.8 for q in kept): kept.append(r)
        recs = finish_recs(kept, tracker, zones, t)
        for r in recs: LIVE_ENGINE.step(r["track"], rule_state(r, r["p_helmet"]), ACTIVITY, t, 1 / fps, clip=f"{out_name}-{k}")
        lat.append((time.perf_counter() - t0) * 1000); counts.append(len(recs))
        if truth is not None and recs:
            gt = np.array(truth[k]); errs += [float(np.min(np.linalg.norm(gt - r["foot"][:2], axis=1))) for r in recs]
        vis = _to3(g.copy())                                                    # annotate on the fisheye frame itself
        for z in zones:
            if z.kind == "polygon": P = np.c_[np.asarray(z.params["poly"], float), np.zeros(len(z.params["poly"]))]; closed = True
            elif z.kind == "sphere":
                c = z.params["center"](t) if callable(z.params["center"]) else z.params["center"]
                re = max(math.sqrt(max(z.params["r"] ** 2 - (hb - c[2]) ** 2, 0)) for hb in (0.1, 0.9, 1.7)); a = np.linspace(0, 2 * math.pi, 64)
                P = np.c_[c[0] + re * np.cos(a), c[1] + re * np.sin(a), np.zeros(64)]; closed = True
            else:
                n = np.asarray(z.params["n"], float); n /= np.linalg.norm(n); d_ = np.array([-n[1], n[0], 0.0])
                P = np.array([n * z.params["d"] + s_ * d_ for s_ in np.linspace(-40, 60, 200)]); closed = False
            px, th = kb_project(fish, P); px = px[th < math.radians(92)]
            if len(px) > 1: cv2.polylines(vis, [px.astype(np.int32)], closed, (0, 200, 255), 3, cv2.LINE_AA)
        for r in recs:
            pf, _ = kb_project(fish, np.r_[r["foot"][:2], 0.0]); bad = any(r["state"]["inside"].values())
            cv2.circle(vis, (int(pf[0, 0]), int(pf[0, 1])), 14, (0, 0, 255) if bad else (0, 255, 0), 3)
        for v in views:                                                         # outline of each virtual view on the fisheye
            w_, h_ = view_size; edge = np.r_[np.c_[np.linspace(0, w_, 20), np.zeros(20)], np.c_[np.full(20, w_), np.linspace(0, h_, 20)],
                                            np.c_[np.linspace(w_, 0, 20), np.full(20, h_)], np.c_[np.zeros(20), np.linspace(h_, 0, 20)]]
            ex = cv2.remap(v["mapx"], edge[:, 0].astype(np.float32).reshape(-1, 1).clip(0, w_ - 1), edge[:, 1].astype(np.float32).reshape(-1, 1).clip(0, h_ - 1), cv2.INTER_LINEAR).ravel()
            ey = cv2.remap(v["mapy"], edge[:, 0].astype(np.float32).reshape(-1, 1).clip(0, w_ - 1), edge[:, 1].astype(np.float32).reshape(-1, 1).clip(0, h_ - 1), cv2.INTER_LINEAR).ravel()
            cv2.polylines(vis, [np.c_[ex, ey].astype(np.int32)], True, (255, 160, 60), 2, cv2.LINE_AA)
        small = cv2.resize(vis, (1440, int(1440 * vis.shape[0] / vis.shape[1])))
        if writer is None: writer = cv2.VideoWriter(str(BASE / f"{out_name}.mp4"), cv2.VideoWriter_fourcc(*"mp4v"), fps, small.shape[1::-1])
        writer.write(small); cv2.imwrite(str(BASE / f"{out_name}.jpg"), small)
    if writer: writer.release()
    return {"views": len(views), "frames": len(counts), "workers_per_frame": f"median {int(np.median(counts))}, min {min(counts)}, max {max(counts)}",
            "position_error_m_median": round(float(np.median(errs)), 3) if errs else None, "ms_per_frame": round(float(np.median(lat)), 1),
            "output": str(BASE / f"{out_name}.mp4"), "last_frame": str(BASE / f"{out_name}.jpg")}

if CFG.fisheye_demo and RIG_DEMO_TRUTH is not None:
    demo_cam = scene_camera("rig_demo", 1920, 1080)
    FISH = make_fisheye([[892, 0, 1440], [0, 892, 1440], [0, 0, 1]], [0.02, -0.01, 0.002, -0.0005], demo_cam.R, demo_cam.C, (2880, 2880))
    fmx, fmy = fisheye_from_pinhole_maps(FISH, demo_cam, (1920, 1080))
    cap = cv2.VideoCapture(str(BASE / "rig_demo.mp4")); fish_frames = []
    while len(fish_frames) < 20:
        ok, f = cap.read()
        if not ok: break
        fish_frames.append(cv2.remap(f, fmx, fmy, cv2.INTER_LINEAR, borderValue=(25, 25, 25)))
    REPORT["fisheye"] = run_fisheye(fish_frames, FISH, [(-30.0, 25.0), (0.0, 25.0), (30.0, 25.0)], SCENES["rig_demo"]["zones"](), truth=RIG_DEMO_TRUTH)
    print(f"[§9b] 185° fisheye (rig demo resampled) → 3 upright virtual views → one tracker: {REPORT['fisheye']}")
    if IN_COLAB:
        cv2_imshow(cv2.imread(REPORT["fisheye"]["last_frame"]))

# ─────────────────────────────── §10 OpenAI-compatible endpoint ──────────────────
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse, StreamingResponse
import uvicorn

MODEL_ID = "vg360-clifford-safety"
app = FastAPI(title="VG360 Clifford safety gateway")

def summarise(question, events):
    q = question.lower(); m = re.search(r"last\s+(\d+)", q); n = int(m.group(1)) if m else 10
    kinds = [k for k in ("fall", "intrusion", "ppe", "rule") if k in q] or ["fall", "intrusion", "ppe", "rule"]
    sel = [e for e in events if any(e["type"].startswith(k) for k in kinds)][-n:]
    counts = {}
    for e in events: counts[e["type"]] = counts.get(e["type"], 0) + 1
    lines = [f"Safety events total: " + (", ".join(f"{k}={v}" for k, v in counts.items()) or "none") + "."]
    for e in sel:
        if e["type"] == "fall":
            lines.append(f"t={e['t']}s FALL track #{e['track']} (head {e['head_z_m']} m, tilt {e['tilt_deg']}°)")
        elif e["type"] == "rule":
            lines.append(f"t={e['t']}s RULE {e['hz'] or e['rule']} ({e['pri'] or '-'}) {e['name']} · track #{e['track']}"
                         + (f" · ledger {e['hash'][:12]}…" if e.get("hash") else ""))
        elif e["type"] == "intrusion":
            lines.append(f"t={e['t']}s INTRUSION track #{e['track']} in {e['zone']} (clearance {e['clearance_m']} m)")
        else:
            lines.append(f"t={e['t']}s NO HELMET track #{e['track']} (p_helmet {e['p_helmet']})")
    return "\n".join(lines)

def _completion(content, model):
    return {"id": f"chatcmpl-{uuid.uuid4().hex[:24]}", "object": "chat.completion", "created": int(time.time()),
            "model": model, "choices": [{"index": 0, "message": {"role": "assistant", "content": content},
                                         "finish_reason": "stop"}],
            "usage": {"prompt_tokens": 0, "completion_tokens": len(content.split()), "total_tokens": len(content.split())}}

@app.get("/v1/rules")
def rules_api():
    return {"rules": REPORT.get("rule_studio", {}).get("rules", {})}

@app.get("/v1/ledger")
def ledger_api():
    led = LIVE_LEDGER if LIVE_LEDGER.entries else SIM_LEDGER
    return {"verified": led.verify(), "entries": led.entries[-200:]}

@app.get("/v1/models")
def models():
    return {"object": "list", "data": [{"id": MODEL_ID, "object": "model", "owned_by": "vg360"}]}

@app.post("/v1/chat/completions")
async def chat(req: Request):
    body = await req.json(); msgs = body.get("messages", []); model = body.get("model", MODEL_ID)
    question = next((m.get("content", "") for m in reversed(msgs) if m.get("role") == "user"), "")
    if isinstance(question, list): question = " ".join(p.get("text", "") for p in question if isinstance(p, dict))
    facts = summarise(question, EVENT_LOG)
    if CFG.upstream_base_url:                                  # optional: real LLM phrases the deterministic facts
        r = requests.post(CFG.upstream_base_url.rstrip("/") + "/chat/completions", timeout=60,
                          headers={"Authorization": f"Bearer {CFG.upstream_api_key}"},
                          json={**body, "stream": False, "model": CFG.upstream_model or model,
                                "messages": [{"role": "system", "content": "Answer only from these safety facts:\n" + facts}] + msgs})
        content = r.json()["choices"][0]["message"]["content"]
    else:
        content = facts
    if body.get("stream"):
        cid = f"chatcmpl-{uuid.uuid4().hex[:24]}"
        def gen():
            for i, tok in enumerate(re.findall(r"\S+\s*", content)):
                delta = {"role": "assistant", "content": tok} if i == 0 else {"content": tok}
                yield "data: " + json.dumps({"id": cid, "object": "chat.completion.chunk", "created": int(time.time()),
                                             "model": model, "choices": [{"index": 0, "delta": delta, "finish_reason": None}]}) + "\n\n"
            yield "data: " + json.dumps({"id": cid, "object": "chat.completion.chunk", "created": int(time.time()), "model": model,
                                         "choices": [{"index": 0, "delta": {}, "finish_reason": "stop"}]}) + "\n\n"
            yield "data: [DONE]\n\n"
        return StreamingResponse(gen(), media_type="text/event-stream")
    return JSONResponse(_completion(content, model))

SERVER = None
if CFG.serve:
    SERVER = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=CFG.port, log_level="warning"))
    threading.Thread(target=SERVER.run, daemon=True).start()
    for _ in range(50):
        try: requests.get(f"http://127.0.0.1:{CFG.port}/v1/models", timeout=1); break
        except Exception: time.sleep(0.2)
    r = requests.post(f"http://127.0.0.1:{CFG.port}/v1/chat/completions",
                      json={"model": MODEL_ID, "messages": [{"role": "user", "content": "List the last 5 falls and intrusions"}]})
    print("[§10] POST /v1/chat/completions →\n" + r.json()["choices"][0]["message"]["content"])
    rs = requests.post(f"http://127.0.0.1:{CFG.port}/v1/chat/completions", stream=True,
                       json={"model": MODEL_ID, "stream": True, "messages": [{"role": "user", "content": "any falls?"}]})
    chunks = [ln for ln in rs.iter_lines(decode_unicode=True) if ln.startswith("data: ")]
    assert chunks[-1] == "data: [DONE]"
    print(f"      stream=True → {len(chunks)} SSE chunks, ends with [DONE] ✓")
    if IN_COLAB:
        from google.colab.output import eval_js
        print("      public proxy URL:", eval_js(f"google.colab.kernel.proxyPort({CFG.port})"))

(BASE / "report.json").write_text(json.dumps(REPORT, indent=2, default=str))
print(f"\n[done] report → {BASE/'report.json'}")
